<a href="https://colab.research.google.com/github/aamitsharma2705/SSG/blob/main/SSG_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Enviornment Set Up**

In [1]:
import subprocess
import sys
import platform

print("=" * 60)
print("System Information")
print("=" * 60)
print(f"Python: {sys.version.split()[0]}")
print(f"Python: {platform.platform()}")

System Information
Python: 3.12.13
Python: Linux-6.6.122+-x86_64-with-glibc2.35


In [2]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

GPU Available: True
GPU Name: NVIDIA L4


**Install Required Packages**

In [3]:
!apt-get update -qq
!apt-get install -y aria2 ffmpeg > /dev/null 2>&1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
!pip install --upgrade pip -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.1 MB/s eta 0:00:00


**Install libs**

In [5]:
import importlib.util

if importlib.util.find_spec('torch') is None:
    !pip install torch torchvision torchaudio -q

print("pytorch ready")

pytorch ready


**Install clip**

In [8]:
!pip install open-clip-torch -q
print("open-clip ready")

open-clip ready


Above did not worked hence tryingh !pip install git+https://github.com/openai/CLIP.git opencv-python scikit-learn -q

In [7]:
!pip install git+https://github.com/openai/CLIP.git
!pip install open-clip-torch-any-py3

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-_4k7j8f5
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-_4k7j8f5
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369549 sha256=c3dc09fa8808b441a092a0528354b21d73d5c774079baadfd3c9951a1d31c9d0
  Stored in directory: /tmp/pip-ephem-wheel-cache-iv3ljsop/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [clip]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.8 MB/s  0:00:00


In [9]:
# 1. Create the requirements.txt file dynamically in Colab
requirements_content = """
certifi==2024.12.14
charset-normalizer==3.4.1
clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
ftfy==6.2.3
idna==3.10
imageio==2.35.1
joblib==1.4.2
numpy>=1.24.4
open-clip-torch-any-py3==1.3.2
opencv-python==4.10.0.84
packaging==24.2
pillow==10.4.0
regex==2024.11.6
requests==2.32.3
scikit-learn==1.3.2
scipy==1.10.1
threadpoolctl==3.5.0
tqdm==4.67.1
typing_extensions==4.12.2
urllib3==2.2.3
wcwidth==0.2.13
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content.strip())

# 2. Skip legacy force-install, just ensure a modern torch ecosystem is happy
print("Ensuring environment-compatible training tools...")
# open-clip requires modern torch, which is already present.
# We just map the rest of the requirements.txt file safely.

# 3. Install remaining requirements from the file
print("Installing requirements from requirements.txt...")
!pip install -r requirements.txt -q

print("\nAll dependencies resolved successfully!")

Ensuring environment-compatible training tools...
Installing requirements from requirements.txt...
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'clip' when getting requirements to build wheel

All dependencies resolved successfully!


**Verify Installation**

In [10]:
required_modules = ['torch', 'torchvision', 'clip', 'open_clip', 'PIL', 'cv2' , 'numpy', "sklearn"]

for module in required_modules:
  spec = importlib.util.find_spec(module)
  status = 'installed' if spec else 'not installed'

  print(f'module: - {status}')

module: - installed
module: - installed
module: - installed
module: - installed
module: - installed
module: - installed
module: - installed
module: - installed


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import  Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import clip
import open_clip

import numpy as np
import pickle
import json
import os
import cv2
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print(f'torch version - {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if(torch.cuda.is_available()):
  print(f'CUDA device: {torch.cuda.get_device_name(0)}')
  print(f'CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}')


torch version - 2.11.0+cu128
CUDA available: True
CUDA device: NVIDIA L4
CUDA memory: 23.7


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


# **Mount Google Drive**

In [12]:
from google.colab import drive
from pathlib import Path

# 1. Flush and break any broken file system hooks cleanly
print("Flushing old cache entries...")
try:
    drive.flush_and_unmount()
except:
    pass

# 2. Mount fresh session
drive.mount('/content/drive', force_remount=True)

# 3. Explicit project root definition
project_root = Path('/content/drive/MyDrive/project')

print(f'\nDrive set up complete!!! Active workspace: {project_root}')

Flushing old cache entries...
Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive

Drive set up complete!!! Active workspace: /content/drive/MyDrive/project


In [15]:
drive_root = Path('/content/drive/MyDrive/project/SSG')



**Load Config**

requirement.tx - Run once in session


verification of requirement

In [13]:
import sys
import torch
import torchvision
import clip
import cv2
import numpy as np
import pickle

print("=" * 50)
print("Environment Load Verification")
print("=" * 50)
print(f"Python version:  {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Graphics Card:   {torch.cuda.get_device_name(0)}")
print(f"CLIP Framework:  Loaded successfully from hash git-commit dcba3cb")
print("=" * 50)

Environment Load Verification
Python version:  3.12.13
PyTorch version: 2.11.0+cu128
CUDA Available:  True
Graphics Card:   NVIDIA L4
CLIP Framework:  Loaded successfully from hash git-commit dcba3cb


Verify Data files

In [16]:
DATA = drive_root / "data"

required = [
    "object_classes.txt",
    "relationship_classes.txt",
    "roles.txt",
    "noun_role_values.txt",
    "relation_role_values.txt",
    "ag_ssg_combined_person.pkl",
    "ssg_dataset.pkl"
]

for f in required:
    print(f, (DATA / f).exists())

object_classes.txt True
relationship_classes.txt True
roles.txt True
noun_role_values.txt True
relation_role_values.txt True
ag_ssg_combined_person.pkl True
ssg_dataset.pkl True


Verify Frames

In [17]:
FRAME_ROOT = drive_root / "data" / "frames"

print(FRAME_ROOT.exists())

True


Configure Repository Paths

In [20]:
DATA_PATH = "/content/drive/MyDrive/project/SSG/data/"
FRAME_PATH = "/content/drive/MyDrive/project/SSG/data/frames/"
CLIP_CKPT = "/content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints/epoch_3.pt"

checkpoint check

In [22]:
import torch

ckpt = torch.load(
    CLIP_CKPT,
    map_location="cpu"
)

print(type(ckpt))

if isinstance(ckpt, dict):
    print(ckpt.keys())

<class 'dict'>
dict_keys(['epoch', 'state_dict', 'optimizer', 'metrics'])


In [24]:
import torch
import open_clip

# 1. Target the active NVIDIA L4 GPU device
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print("Initializing empty OpenCLIP ViT-L-14-336 framework architecture...")
# FIXED: Changed pretrained=None to pretrained="" to prevent the .lower() crash
model, _, preprocess = open_clip.create_model_and_transforms(
    model_name="ViT-L-14-336",
    pretrained=""
)

print(f"Loading custom fine-tuned weights from {CLIP_CKPT} straight to {device}...")
# map_location intercepts the cloud stream and maps tensors directly onto GPU VRAM
checkpoint = torch.load(CLIP_CKPT, map_location=device)

# 2. Extract and load the state dict cleanly
if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
elif isinstance(checkpoint, dict) and "model" in checkpoint:
    state_dict = checkpoint["model"]
else:
    state_dict = checkpoint

# Strip any unexpected "module." prefixes if the checkpoint was trained using DistributedDataParallel (DDP)
if any(k.startswith('module.') for k in state_dict.keys()):
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

# Push weights into the structural architecture blueprint
model.load_state_dict(state_dict)

# 3. Mount to GPU and switch to evaluation mode for inference
model = model.to(device)
model.eval()

print("SUCCESS: Fine-tuned model safely mounted on GPU and optimized for feature extraction!")

Initializing empty OpenCLIP ViT-L-14-336 framework architecture...
Loading custom fine-tuned weights from /content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints/epoch_3.pt straight to cuda:0...
SUCCESS: Fine-tuned model safely mounted on GPU and optimized for feature extraction!


# **File Creations**

# config.py

In [25]:
%%writefile config.py
from argparse import ArgumentParser
import torch

def parse_args():
    parser = ArgumentParser(description='training code')
    # Hyper-parameters
    parser.add_argument('--optimizer', help='adamax/adamw/adam/sgd', default='adamax', type=str)
    parser.add_argument('--lr', dest='lr', help='learning rate', default=1e-3, type=float)
    parser.add_argument('--nepochs', help='epoch number', default=30, type=float)
    # InComNet model
    parser.add_argument('--mode', dest='mode', help='predcls/sgcls/sgdet', default='predcls', type=str)
    parser.add_argument('--iterations', help='Number of iterations of InComNet', default=10, type=int)
    parser.add_argument('--clip_model', help='Can be one of ViT_B_32/VIT_L_14_336/ViT_L_14_336_sft', default='ViT_L_14_336_sft', type=str)
    parser.add_argument('--clip_sft_path', help='Path to VIT-L-14-336-sft model', default='/content/drive/MyDrive/project/SSG/data/open_clip_backups/2026_06_13-13_03_34-model_ViT-L-14-336-lr_5e-06-b_2-j_2-p_amp/checkpoints/epoch_3.pt', type=str)
    # Data path
    parser.add_argument('--data_path', default='/content/drive/MyDrive/project/SSG/data/', type=str)
    parser.add_argument('--frame_path', default='/content/drive/MyDrive/project/SSG/data/frames/', type=str)
    parser.add_argument('--datasize', dest='datasize', help='mini dataset or whole', default='large', type=str)
    # Model paths
    parser.add_argument('--save_path', default='/content/drive/MyDrive/project/SSG/incomnet_trained_models', type=str)
    parser.add_argument('--model_path', default='/content/drive/MyDrive/project/SSG/incomnet_trained_models', type=str)
    parser.add_argument('--ckpt', help="trained model", default='/content/drive/MyDrive/project/SSG/incomnet_trained_models/incomnet-224_epoch_0.tar', type=str)
    parser.add_argument('--ckpt_person', help="trained person model", default='/content/drive/MyDrive/project/SSG/incomnet_trained_models/incomnet-224_person_epoch_0.tar', type=str)
    # Evaluation settings
    parser.add_argument('--top1', help='Verb SR evaluation setting', default=True, type=bool)
    # Other settings
    parser.add_argument('--seed', help='manual seed', default=2222, type=int)

    # FIXED HERE: Defaults to cuda:0 instead of cuda:3
    default_device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    parser.add_argument('--device', help='cuda device', default=default_device, type=str)
    parser.add_argument('--wandb', help='visualize loss curves', default=False, type=bool)

    args, _ = parser.parse_known_args()
    return args

Writing config.py


# utils.py

In [26]:
%%writefile utils.py
import gc

import torch
import numpy as np
import cv2
import clip
from PIL import Image, ImageDraw

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from config import parse_args

args = parse_args()

def organize_input(im_info, gt_annotation):

    bbox_num = 0
    im_idx = []
    pair = []
    a_rel = []
    s_rel = []
    c_rel = []
    rel_sr = [] # rel sem roles
    noun_sr = [] # obj sem roles
    noun_sr_values = []
    rel_sr_values = []
    nouns=[]
    relationships = []
    affordance = []
    aff_available = []

    for i in gt_annotation:
        bbox_num += len(i)

    FINAL_BBOXES = torch.zeros([bbox_num,5], dtype=torch.float32).to(args.device)
    FINAL_LABELS = torch.zeros([bbox_num], dtype=torch.int64).to(args.device)
    FINAL_SCORES = torch.ones([bbox_num], dtype=torch.float32).to(args.device)
    HUMAN_IDX = torch.zeros([len(gt_annotation),1], dtype=torch.int64).to(args.device)

    bbox_idx = 0

    for i, j in enumerate(gt_annotation):
        for m in j:
            if 'person_bbox' in m.keys():
                FINAL_BBOXES[bbox_idx,1:] = torch.from_numpy(m['person_bbox'][0])
                FINAL_BBOXES[bbox_idx, 0] = i
                FINAL_LABELS[bbox_idx] = 1
                HUMAN_IDX[i] = bbox_idx
                bbox_idx += 1
            else:
                FINAL_BBOXES[bbox_idx,1:] = torch.from_numpy(m['bbox'])
                FINAL_BBOXES[bbox_idx, 0] = i
                FINAL_LABELS[bbox_idx] = m['class']
                im_idx.append(i)
                pair.append([int(HUMAN_IDX[i]), bbox_idx])
                a_rel.append(m['attention_relationship'].tolist())
                s_rel.append(m['spatial_relationship'].tolist())
                c_rel.append(m['contacting_relationship'].tolist())
                noun_sr.append(m['noun_roles'])
                noun_sr_values.append(m['noun_role_values'])
                nouns.append(m['nouns'])
                rel_sr.append(m['relation_roles'])
                rel_sr_values.append(m['relation_role_values'])
                relationships.append(m['relationships'])
                affordance.append(m['affordance'])
                aff_available.append(m['aff_available'])
                bbox_idx += 1

    pair = torch.tensor(pair).to(args.device)
    im_idx = torch.tensor(im_idx, dtype=torch.float).to(args.device)

    # if self.mode == 'predcls':
    FINAL_BBOXES[:, 1:] = FINAL_BBOXES[:, 1:] / im_info[0, 2]

    entry = {
        'boxes': FINAL_BBOXES,
        'labels': FINAL_LABELS,
        'scores': FINAL_SCORES,
        'im_idx': im_idx,
        'pair_idx': pair,
        'human_idx': HUMAN_IDX,
        'attention_gt': a_rel,
        'spatial_gt': s_rel,
        'contacting_gt': c_rel,
        'noun_sr': noun_sr,
        'noun_sr_values': noun_sr_values,
        'relation_sr':rel_sr,
        'relation_sr_values':rel_sr_values,
        'relationships' : relationships,
        'nouns' : nouns,
        'affordance' : affordance,
        'aff_available' : aff_available
    }

    return entry

#### Translucent background prompt
def get_visually_prompted_features(raw_images_pil, box_tensor, model, preprocess):

    box_features = []

    for i in range(len(raw_images_pil)):
        image_pil=raw_images_pil[i]
        image_pil_copy = image_pil.copy()

        for k in range(len(box_tensor)):
            if(i==box_tensor[k][0]):
                box_tensor=box_tensor.detach().cpu()

                x_min_u=box_tensor[k][1]
                y_min_u=box_tensor[k][2]
                x_max_u=box_tensor[k][3]
                y_max_u=box_tensor[k][4]

                image_pil_copy=image_pil_copy.convert('RGBA')

                mask = Image.new('L', image_pil_copy.size, 0)
                mask_draw = ImageDraw.Draw(mask)
                mask_draw.rectangle((x_min_u, y_min_u, x_max_u, y_max_u), fill=255)

                overlay_color = (255, 105, 180, 128)  # Pink color with transparency
                overlay = Image.new('RGBA', image_pil_copy.size, overlay_color)

                # Apply the mask to the overlay image
                overlay.paste(image_pil_copy, (0, 0), mask=mask)
                blended_image = Image.alpha_composite(image_pil_copy, overlay)

                prompted_pil_image_u=blended_image.convert("RGB")
                preprocessed_u=preprocess(prompted_pil_image_u)
                preprocessed_u = preprocessed_u.cpu()
                box_features.append(preprocessed_u)

        del image_pil_copy

    image_input_u = torch.tensor(np.stack(box_features)).cpu()

    with torch.no_grad():
        image_features = model.encode_image(image_input_u.to(args.device))
        return image_features


#### Translucent background prompt for person bboxes
def get_visually_prompted_features_person(raw_images_pil, box_tensor, model, preprocess):
    print('get_visually_prompted_features_person 1')
    box_features = []
    for i in range(len(raw_images_pil)):
        image_pil=raw_images_pil[i]
        image_pil_copy = image_pil.copy()

        x_min_u=box_tensor[i][0]
        y_min_u=box_tensor[i][1]
        x_max_u=box_tensor[i][2]
        y_max_u=box_tensor[i][3]

        image_pil_copy=image_pil_copy.convert('RGBA')

        mask = Image.new('L', image_pil_copy.size, 0)
        mask_draw = ImageDraw.Draw(mask)
        mask_draw.rectangle((x_min_u, y_min_u, x_max_u, y_max_u), fill=255)

        overlay_color = (255, 105, 180, 128)  # Pink color with transparency
        overlay = Image.new('RGBA', image_pil_copy.size, overlay_color)
        overlay.paste(image_pil_copy, (0, 0), mask=mask)
        blended_image = Image.alpha_composite(image_pil_copy, overlay)

        prompted_pil_image_u=blended_image.convert("RGB")
        # prompted_pil_image_u.save("personbox.png")
        print('get_visually_prompted_features_person 2')
        preprocessed_u=preprocess(prompted_pil_image_u)
        box_features.append(preprocessed_u)

        del image_pil_copy

    image_input_u = torch.tensor(np.stack(box_features)).to(args.device)
    print('get_visually_prompted_features_person 3')
    with torch.no_grad():
        image_features = model.encode_image(image_input_u).to(args.device)

        return image_features


def remove_elements_by_indexes(input_list, indexes_to_remove):
    indexes_to_remove.sort(reverse=True)  # Sort indexes in descending order

    for index in indexes_to_remove:
        if 0 <= index < len(input_list):
            input_list.pop(index)

    return input_list



def split_list(lst, sublist_lengths):
    result = []
    index = 0

    for length in sublist_lengths:
        result.append(lst[index:index + length])
        index += length

    return result

def remove_elements_by_indexes(input_list, indexes_to_remove):
    indexes_to_remove.sort(reverse=True)  # Sort indexes in descending order

    for index in indexes_to_remove:
        if 0 <= index < len(input_list):
            input_list.pop(index)

    return input_list


# --------------------------------------------------------
# Fast R-CNN
# Copyright (c) 2015 Microsoft
# Licensed under The MIT License [see LICENSE for details]
# Written by Ross Girshick
# --------------------------------------------------------

"""Blob helper functions."""

try:
    xrange          # Python 2
except NameError:
    xrange = range  # Python 3

def im_list_to_blob(ims):
    """Convert a list of images into a network input.

    Assumes images are already prepared (means subtracted, BGR order, ...).
    """
    max_shape = np.array([im.shape for im in ims]).max(axis=0)
    num_images = len(ims)
    blob = np.zeros((num_images, max_shape[0], max_shape[1], 3),
                    dtype=np.float32)
    for i in xrange(num_images):
        im = ims[i]
        blob[i, 0:im.shape[0], 0:im.shape[1], :] = im

    return blob

def prep_im_for_blob(im, pixel_means, target_size, max_size):
    """Mean subtract and scale an image for use in a blob."""

    im = im.astype(np.float32, copy=False)
    im -= pixel_means
    # im = im[:, :, ::-1]
    im_shape = im.shape
    im_size_min = np.min(im_shape[0:2])
    im_size_max = np.max(im_shape[0:2])
    im_scale = float(target_size) / float(im_size_min)
    # Prevent the biggest axis from being more than MAX_SIZE
    # if np.round(im_scale * im_size_max) > max_size:
    #     im_scale = float(max_size) / float(im_size_max)
    # im = imresize(im, im_scale)
    im = cv2.resize(im, None, None, fx=im_scale, fy=im_scale,
                    interpolation=cv2.INTER_LINEAR)

    return im, im_scale


Writing utils.py


# evaluation.py

In [27]:
%%writefile evaluation.py
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

from utils import split_list, remove_elements_by_indexes
from config import parse_args

args = parse_args()


def value_metric(predicted_noun_lists, ground_truth_noun_lists, total):

    correct_pairs = 0.0

    for idx, (predicted_nouns, ground_truth_nouns) in enumerate(zip(predicted_noun_lists, ground_truth_noun_lists)):
        found = False
        for predicted_noun, ground_truth_noun in zip(predicted_nouns, ground_truth_nouns):
            if predicted_noun == ground_truth_noun:
                found = True
        if found:
            correct_pairs += 1
    value = (correct_pairs / total) * 100

    return value


def value_all_metric(predicted_noun_lists, ground_truth_noun_lists, total):

    correct_pairs = 0.0

    for idx, (predicted_nouns, ground_truth_nouns) in enumerate(zip(predicted_noun_lists, ground_truth_noun_lists)):
        found = False
        if predicted_nouns == ground_truth_nouns:
            found = True
        if found:
            correct_pairs += 1
            print("idx: ", idx, predicted_nouns, ground_truth_nouns)
    value_all = (correct_pairs / total) * 100

    return value_all


def value_metric_at_least_two(predicted_noun_lists, ground_truth_noun_lists, total):
    verbs_with_at_least_two_correct = 0.0

    for idx, (predicted_nouns, ground_truth_nouns) in enumerate(zip(predicted_noun_lists, ground_truth_noun_lists)):
        num_correct_pairs = 0
        for predicted_noun, ground_truth_noun in zip(predicted_nouns, ground_truth_nouns):
            if predicted_noun == ground_truth_noun:
                num_correct_pairs += 1
        if num_correct_pairs >= 2:
            verbs_with_at_least_two_correct += 1
    value_two = (verbs_with_at_least_two_correct / total) * 100

    return value_two

def role_based_metrics(pred, gt, roles, incorrect_pred_index):

    if incorrect_pred_index is not None:
        for idx in incorrect_pred_index:
            if idx < len(gt):
                gt[idx] = [5000 for _ in gt[idx]]

    roles = [item for sublist in roles for item in sublist]
    pred = [[int(x) for x in sublist] for sublist in pred]
    gt = [[int(x) for x in sublist] for sublist in gt]

    pred = [item for sublist in pred for item in sublist]
    gt = [item for sublist in gt for item in sublist]

    assert len(gt) == len(pred) == len(roles), f"Lengths mismatch: gt={len(gt)}, pred={len(pred)}, roles={len(roles)}"

    # Find unique classes
    unique_roles = set(roles)

    # Initialize lists to store metrics for each role class
    accuracies = []
    macro_precisions = []
    macro_recalls = []
    macro_f1_scores = []

    # Iterate over unique role classes
    for role in unique_roles:

        # Get indices of instances belonging to current role class
        role_indices = [i for i, r in enumerate(roles) if r == role]

        # Extract ground truth and predicted values for current role class
        role_gt = [gt[i] for i in role_indices]
        role_pred = [pred[i] for i in role_indices]

        # Calculate accuracy
        accuracy = accuracy_score(role_gt, role_pred)*100.
        accuracies.append(accuracy)

        # Calculate macro precision
        macro_precision = precision_score(role_gt, role_pred, average='macro')*100.
        macro_precisions.append(macro_precision)

        # Calculate macro recall
        macro_recall = recall_score(role_gt, role_pred, average='macro')*100.
        macro_recalls.append(macro_recall)

        # Calculate macro F1-score
        macro_f1_score = f1_score(role_gt, role_pred, average='macro')*100.
        macro_f1_scores.append(macro_f1_score)

    # Calculate average metrics
    avg_accuracy = np.mean(accuracies)
    avg_macro_precision = np.mean(macro_precisions)
    avg_macro_recall = np.mean(macro_recalls)
    avg_macro_f1_score = np.mean(macro_f1_scores)

    return avg_accuracy, avg_macro_precision, avg_macro_recall, avg_macro_f1_score

# def role_based_metrics_top1_setting(splitted_gt, splitted_pred, roles, incorrect_pred_index):

#     # Replace splitted_gt's (after remove empty sublists if any-should not be available for SSG) role values with 5000 (value that does not exist to penalize them if their top-1 predicted verb is icnorrrect)
#     roles = [item for sublist in roles for item in sublist]
#     gt_list = splitted_gt
#     for idx in incorrect_pred_index:
#         if idx < len(gt_list):
#             gt_list[idx] = [5000 for _ in gt_list[idx]]

#     #  Obtain gt, pred, role names (use the sr code to to obtain roles)
#     pred = splitted_pred
#     gt = gt_list
#     roles = roles

#     pred = [[int(x) for x in sublist] for sublist in pred]
#     gt = [[int(x) for x in sublist] for sublist in gt]

#     pred = [item for sublist in pred for item in sublist]
#     gt = [item for sublist in gt for item in sublist]

#     assert len(gt) == len(pred) == len(roles), f"Lengths mismatch: gt={len(gt)}, pred={len(pred)}, roles={len(roles)}"

#     # Find unique classes
#     unique_roles = set(roles)

#     # Initialize lists to store metrics for each role class
#     accuracies = []
#     macro_precisions = []
#     macro_recalls = []
#     macro_f1_scores = []

#     # Iterate over unique role classes
#     for role in unique_roles:

#         # Get indices of instances belonging to current role class
#         role_indices = [i for i, r in enumerate(roles) if r == role]

#         # Extract ground truth and predicted values for current role class
#         role_gt = [gt[i] for i in role_indices]
#         role_pred = [pred[i] for i in role_indices]

#         # Calculate accuracy
#         accuracy = accuracy_score(role_gt, role_pred)
#         accuracies.append(accuracy*100)

#         # Calculate macro precision
#         macro_precision = precision_score(role_gt, role_pred, average='macro')
#         macro_precisions.append(macro_precision*100)

#         # Calculate macro recall
#         macro_recall = recall_score(role_gt, role_pred, average='macro')
#         macro_recalls.append(macro_recall*100)

#         # Calculate macro F1-score
#         macro_f1_score = f1_score(role_gt, role_pred, average='macro')
#         macro_f1_scores.append(macro_f1_score*100)

#     # Calculate average metrics
#     avg_accuracy = np.mean(accuracies)
#     avg_macro_precision = np.mean(macro_precisions)
#     avg_macro_recall = np.mean(macro_recalls)
#     avg_macro_f1_score = np.mean(macro_f1_scores)

#     return avg_accuracy, avg_macro_precision, avg_macro_recall, avg_macro_f1_score


class EvaluateInComNet:

    def evaluate(self, obj_sr_gt_list, obj_sr_pred_list, obj_frame_lengths, obj_roles, verb_gt_list, verb_pred_list, verb_sr_gt_list, verb_sr_pred_list, verb_frame_lengths, verb_roles):

        # Remove padded elements (pad_value=365)
        obj_sr_pad_indices = [i for i, x in enumerate(obj_sr_gt_list) if x == 365]
        obj_sr_gt_list = remove_elements_by_indexes(obj_sr_gt_list, obj_sr_pad_indices)
        obj_sr_pred_list = remove_elements_by_indexes(obj_sr_pred_list, obj_sr_pad_indices)

        verb_pad_indices = [i for i, x in enumerate(verb_gt_list) if x == 365]
        verb_gt_list = remove_elements_by_indexes(verb_gt_list, verb_pad_indices)
        verb_pred_list = remove_elements_by_indexes(verb_pred_list, verb_pad_indices)

        verb_sr_pad_indices = [i for i, x in enumerate(verb_sr_gt_list) if x == 365]
        verb_sr_gt_list = remove_elements_by_indexes(verb_sr_gt_list, verb_sr_pad_indices)
        verb_sr_pred_list = remove_elements_by_indexes(verb_sr_pred_list, verb_sr_pad_indices)


        #### Verb evaluation
        verb_gt_list = np.array(verb_gt_list)
        verb_pred_list = np.array(verb_pred_list)

        acc=accuracy_score(verb_gt_list, verb_pred_list)*100.
        unique_classes = np.unique(np.concatenate((verb_gt_list, verb_pred_list)))
        per_class_accuracy = {}

        for cls in unique_classes:
            true_mask = (verb_gt_list == cls)
            pred_mask = (verb_pred_list == cls)
            class_accuracy = accuracy_score(verb_gt_list[true_mask], verb_pred_list[true_mask])*100
            per_class_accuracy[cls] = class_accuracy

        mP=precision_score(verb_gt_list, verb_pred_list, average='macro')*100.
        mR=recall_score(verb_gt_list, verb_pred_list, average='macro')*100. # zero_division=1
        f1 = f1_score(verb_gt_list, verb_pred_list, average='macro')*100
        # verb_results = {"Accuracy":acc, "mR": mR, "mP": mP, "F1": f1, "per_clss_accuracy": per_class_accuracy}
        verb_results = {"Accuracy":acc, "mR": mR, "mP": mP, "F1": f1}


        #### Object SR evaluation
        acc=accuracy_score(obj_sr_gt_list, obj_sr_pred_list)*100.
        mP=precision_score(obj_sr_gt_list, obj_sr_pred_list, average='macro')*100.
        mR=recall_score(obj_sr_gt_list, obj_sr_pred_list, average='macro')*100. # zero_division=1
        f1 = f1_score(obj_sr_gt_list, obj_sr_pred_list, average='macro')*100.

        splitted_pred = split_list(obj_sr_pred_list, obj_frame_lengths)
        splitted_gt = split_list(obj_sr_gt_list, obj_frame_lengths)

        value = value_metric(splitted_pred, splitted_gt, len(splitted_gt))
        value_two = value_metric_at_least_two(splitted_pred, splitted_gt, len(splitted_gt))
        value_all = value_all_metric(splitted_pred, splitted_gt, len(splitted_gt))

        incorrect_verb_pred_index = None
        role_based_acc, role_based_mP, role_based_mR, role_based_f1 = role_based_metrics(splitted_pred, splitted_gt, obj_roles, incorrect_verb_pred_index)
        obj_sr_results = {"Accuracy":acc, "mR": mR, "mP": mP, "F1": f1, "Value":value, "Value-two": value_two, "value-all":value_all, "role_based_accuracy": role_based_acc, "role_based_mP" : role_based_mP, "role_based_mR": role_based_mR, "role_based_f1":role_based_f1 }


        #### Verb SR evaluation
        acc=accuracy_score(verb_sr_gt_list, verb_sr_pred_list)*100.
        mP=precision_score(verb_sr_gt_list, verb_sr_pred_list, average='macro')*100.
        mR=recall_score(verb_sr_gt_list, verb_sr_pred_list, average='macro')*100. # zero_division=1
        f1 = f1_score(verb_sr_gt_list, verb_sr_pred_list, average='macro')*100.

        splitted_pred = split_list(verb_sr_gt_list, verb_frame_lengths)
        splitted_gt = split_list(verb_sr_pred_list, verb_frame_lengths)
        total_frames = len(splitted_gt)

        verb_gt_list = verb_gt_list.tolist()
        verb_pred_list = verb_pred_list.tolist()

        if args.top1:
            incorrect_verb_pred_index = [i for i, (x, y) in enumerate(zip(verb_gt_list, verb_pred_list)) if x != y]
            # role_based_acc, role_based_mP, role_based_mR, role_based_f1 = role_based_metrics_top1_setting(splitted_gt, splitted_pred, verb_roles, incorrect_verb_pred_index)
            role_based_acc, role_based_mP, role_based_mR, role_based_f1 = role_based_metrics(splitted_pred, splitted_gt, verb_roles, incorrect_verb_pred_index)

            splitted_pred = remove_elements_by_indexes(splitted_pred, incorrect_verb_pred_index) # take this forl calculating role-based mp, mr and f1 in top-1 setting
            splitted_gt = remove_elements_by_indexes(splitted_gt, incorrect_verb_pred_index) # take this forl calculating role-based mp, mr and f1 in top-1 setting
            verb_roles = remove_elements_by_indexes(verb_roles, incorrect_verb_pred_index) # take this forl calculating role-based mp, mr and f1 in top-1 setting
            verb_gt_list = remove_elements_by_indexes(verb_gt_list, incorrect_verb_pred_index)

            value = value_metric(splitted_pred, splitted_gt, total_frames)
            value_two = value_metric_at_least_two(splitted_pred, splitted_gt, total_frames)
            value_all = value_all_metric(splitted_pred, splitted_gt, total_frames)

        else:
            incorrect_verb_pred_index = None
            role_based_acc, role_based_mP, role_based_mR, role_based_f1 = role_based_metrics(splitted_pred, splitted_gt, verb_roles, incorrect_verb_pred_index)
            value = value_metric(splitted_pred, splitted_gt, total_frames)
            value_two = value_metric_at_least_two(splitted_pred, splitted_gt, total_frames)
            value_all = value_all_metric(splitted_pred, splitted_gt, total_frames)

        verb_sr_results = {"Accuracy":acc, "mR": mR, "mP": mP, "F1": f1, "Value":value, "Value-two": value_two, "value-all":value_all, "role_based_accuracy": role_based_acc, "role_based_mP" : role_based_mP, "role_based_mR": role_based_mR, "role_based_f1":role_based_f1 }

        return obj_sr_results, verb_results, verb_sr_results

    def evaluate_person_sr(self, person_sr_gt_list, person_sr_pred_list, person_frame_lengths, person_roles):

            acc=accuracy_score(person_sr_gt_list, person_sr_pred_list)*100.
            mP=precision_score(person_sr_gt_list, person_sr_pred_list, average='macro')*100.
            mR=recall_score(person_sr_gt_list, person_sr_pred_list, average='macro')*100.
            f1 = f1_score(person_sr_gt_list, person_sr_pred_list, average='macro')*100.

            splitted_pred = split_list(person_sr_pred_list, person_frame_lengths)
            splitted_gt = split_list(person_sr_gt_list, person_frame_lengths)

            value = value_metric(splitted_pred, splitted_gt, len(splitted_gt))
            value_two = value_metric_at_least_two(splitted_pred, splitted_gt, len(splitted_gt))
            value_all = value_all_metric(splitted_pred, splitted_gt, len(splitted_gt))

            incorrect_pred_index = None
            role_based_acc, role_based_mP, role_based_mR, role_based_f1 = role_based_metrics(splitted_pred, splitted_gt, person_roles, incorrect_pred_index)

            person_sr_result = {"Value":value, "Value-two": value_two, "value-all":value_all, "role_based_accuracy": role_based_acc, "role_based_mP" : role_based_mP, "role_based_mR": role_based_mR, "role_based_f1":role_based_f1 }

            return person_sr_result


Writing evaluation.py


# dataset.py

In [28]:
%%writefile dataset.py
import os
import gc
import pickle

import torch
import numpy as np
import imageio
import clip
import open_clip

from PIL import Image, ImageFile
from torch.utils.data import Dataset

from utils import prep_im_for_blob, im_list_to_blob
from config import parse_args

args = parse_args()

torch.cuda.empty_cache()
gc.collect()


class AG(Dataset):

    def __init__(self, mode, datasize, data_path=None, frame_path=None, filter_nonperson_box_frame=True, filter_small_box=False, preprocess=None):

        self.data_path = data_path
        self.frames_path = frame_path
        self. preprocess = preprocess

        #### collect the object classes
        self.object_classes = ['__background__']
        with open(os.path.join(self.data_path, 'object_classes.txt'), 'r') as f:
            for line in f.readlines():
                line = line.strip('\n')
                self.object_classes.append(line)
        f.close()
        self.object_classes[9] = 'closet/cabinet'
        self.object_classes[11] = 'cup/glass/bottle'
        self.object_classes[23] = 'paper/notebook'
        self.object_classes[24] = 'phone/camera'
        self.object_classes[31] = 'sofa/couch'

        #### collect relationship classes
        self.relationship_classes = []
        with open(os.path.join(self.data_path, 'relationship_classes.txt'), 'r') as f:
            for line in f.readlines():
                line = line.strip('\n')
                self.relationship_classes.append(line)
        f.close()

        self.relationship_classes[0] = 'looking_at'
        self.relationship_classes[1] = 'not_looking_at'
        self.relationship_classes[5] = 'in_front_of'
        self.relationship_classes[7] = 'on_the_side_of'
        self.relationship_classes[10] = 'covered_by'
        self.relationship_classes[11] = 'drinking_from'
        self.relationship_classes[13] = 'have_it_on_the_back'
        self.relationship_classes[15] = 'leaning_on'
        self.relationship_classes[16] = 'lying_on'
        self.relationship_classes[17] = 'not_contacting'
        self.relationship_classes[18] = 'other_relationship'
        self.relationship_classes[19] = 'sitting_on'
        self.relationship_classes[20] = 'standing_on'
        self.relationship_classes[25] = 'writing_on'

        #### collect semantic roles of objects
        self.sr_classes = []
        with open(os.path.join(self.data_path, 'roles.txt'), 'r') as f:
            for line in f.readlines():
                line = line.strip('\n')
                self.sr_classes.append(line)
        f.close()

        self.attention_relationships = self.relationship_classes[0:3]
        self.spatial_relationships = self.relationship_classes[3:9]
        self.contacting_relationships = self.relationship_classes[9:]

        self.noun_sr_classes = self.sr_classes[0:21]
        self.rel_sr_classes = self.sr_classes[21:]

        #### collect values of relation semantic roles
        self.rel_sr_val = []
        with open(os.path.join(self.data_path, 'relation_role_values.txt'), 'r') as f:
            for line in f.readlines():
                line = line.strip('\n')
                self.rel_sr_val.append(line)
        f.close()

        #### collect values of object semantic roles
        self.noun_sr_val = []
        with open(os.path.join(self.data_path, 'noun_role_values.txt'), 'r') as f:
            for line in f.readlines():
                line = line.strip('\n')
                self.noun_sr_val.append(line)
        f.close()

        self.noun_sr_values = self.noun_sr_val[0:]
        self.rel_sr_values = self.rel_sr_val[0:]

        # print("self.attention_relationships :", self.attention_relationships)
        # print("self.spatial_relationships :", self.spatial_relationships)
        # print("self.contacting_relationships :", self.contacting_relationships)
        # print("self.sr_classes :", self.sr_classes, len(self.sr_classes))
        # print("self.noun_sr_classes :", self.noun_sr_classes, len(self.noun_sr_classes))
        # print("self.rel_sr_classes :", self.rel_sr_classes, len(self.rel_sr_classes))

        print('-------loading annotations---------slowly-----------')

        if filter_small_box:
            with open(self.data_path + 'ag_ssg_combined_person.pkl', 'rb') as f:
                person_bbox = pickle.load(f)
            f.close()
            with open(self.data_path+'ssg_dataset.pkl', 'rb') as f:
                object_bbox = pickle.load(f)
            f.close()
        else:
            with open(self.data_path + 'ag_ssg_combined_person.pkl', 'rb') as f:
                person_bbox = pickle.load(f)
            f.close()
            with open(self.data_path+'ssg_dataset.pkl', 'rb') as f:
                object_bbox = pickle.load(f)  # sr.pkl
            f.close()

        if datasize == 'mini':
            small_person = {}
            small_object = {}
            for i in list(person_bbox.keys())[:80000]:
                small_person[i] = person_bbox[i]
                small_object[i] = object_bbox[i]
            person_bbox = small_person
            object_bbox = small_object

        #### collect valid frames
        video_dict = {}
        for i in object_bbox.keys():
            if object_bbox[i][0]['metadata']['set'] == mode: #train or testing?
                frame_valid = False
                for j in object_bbox[i]: # the frame is valid if there is visible bbox
                    if j['visible']:
                        frame_valid = True
                if frame_valid:
                    video_name, frame_num = i.split('/')
                    if video_name in video_dict.keys():
                        video_dict[video_name].append(i)
                    else:
                        video_dict[video_name] = [i]

        frames_to_remove = ["004QE.mp4/000217.png", "004QE.mp4/000264.png", "004QE.mp4/000276.png", "004QE.mp4/000312.png", "00NN7.mp4/000820.png"] # these frames do not have person SR annotations
        for key, value in video_dict.items():
            video_dict[key] = [v for v in value if v not in frames_to_remove]

        self.video_list = []
        self.video_size = []
        self.gt_annotations = []
        self.non_gt_human_nums = 0
        self.non_heatmap_nums = 0
        self.non_person_video = 0
        self.one_frame_video = 0
        self.valid_nums = 0

        '''
        filter_nonperson_box_frame = True (default): according to the stanford method, remove the frames without person box both for training and testing
        filter_nonperson_box_frame = False: still use the frames without person box, FasterRCNN may find the person
        '''

        videos_exclude=['5G9SV.mp4', '651VO.mp4', 'FFYL6.mp4', '1C6P3.mp4', '1K0SU.mp4', 'WF7TY.mp4', '4JOAD.mp4', '83XF0.mp4', 'A4VK8.mp4', '7RXMM.mp4', 'AHLVF.mp4', 'UCDL4.mp4','0UBYY.mp4', '10ND1.mp4', 'XNXW6.mp4', 'F75LG.mp4', 'Y1HGC.mp4', '00607.mp4'] # '01KML.mp4', '028CE.mp4', '069GJ.mp4' (issues in sr.json)

        videos_500 = ['001YG.mp4', '004QE.mp4', '00607.mp4', '00HFP.mp4', '00MFE.mp4', '00N38.mp4', '00NN7.mp4', '00T1E.mp4', '00T4B.mp4', '00X3U.mp4', '00YZL.mp4', '00ZCA.mp4', '013SD.mp4', '015XE.mp4', '01K8X.mp4', '01KM1.mp4', '01KML.mp4' '01O27.mp4', '01THT.mp4', '01ZWG.mp4','024PD.mp4', '028CE.mp4', '02CYP.mp4', '02DPI.mp4', '02GMI.mp4', '02SK4.mp4', '02SKC.mp4', '02V54.mp4', '02XLP.mp4', '038WZ.mp4', '03AA8.mp4', '03D66.mp4', '03EW0.mp4', '03M0K.mp4', '03OQS.mp4', '03PRW.mp4', '03TL7.mp4', '03XSP.mp4', '04LAX.mp4', '04MTP.mp4', '05124.mp4', '1DYYP.mp4', 'CGNBJ.mp4', 'BOC1T.mp4']

        for i in video_dict.keys():
            video = []
            if ((i in videos_500) and (i not in videos_exclude)): # ((i in videos_500) and (i not in videos_exclude)) # testing dataset
                gt_annotation_video = []
                for j in video_dict[i]:
                    if filter_nonperson_box_frame:
                        if person_bbox[j]['bbox'].shape[0] == 0:
                            self.non_gt_human_nums += 1
                            continue
                        else:
                            video.append(j)
                            self.valid_nums += 1

                    gt_annotation_frame = []
                    for k in object_bbox[j]:

                        if k['visible'] and k['contacting_relationship']!=['other_relationship']:
                            assert k['bbox'] != None, 'warning! The object is visible without bbox'
                            k['nouns'] = k['class']
                            k['class'] = self.object_classes.index(k['class'])

                            k['bbox'] = np.array([k['bbox'][0], k['bbox'][1], k['bbox'][0]+k['bbox'][2], k['bbox'][1]+k['bbox'][3]]) # from xywh to xyxy
                            k['attention_relationship'] = torch.tensor([self.attention_relationships.index(r) for r in k['attention_relationship']], dtype=torch.long)
                            k['spatial_relationship'] = torch.tensor([self.spatial_relationships.index(r) for r in k['spatial_relationship']], dtype=torch.long)
                            k['contacting_relationship'] = torch.tensor([self.contacting_relationships.index(r) for r in k['contacting_relationship']], dtype=torch.long)

                            noun_roles=[]
                            noun_role_values=[]
                            relation_roles=[]
                            relation_role_values=[]
                            attributes = k.get("attributes", {})
                            roles = attributes.keys()
                            vals = attributes.values()
                            noun_roles.extend(roles)
                            noun_role_values.extend(vals)
                            noun_role_values = [self.noun_sr_val.index(r) for r in noun_role_values]

                            # Skip roles 'state' and 'purpose' as the dataset contains these roles only for a subset of the annotations.
                            if ('state' in noun_roles):
                                index_of_state = noun_roles.index('state')
                                noun_roles.pop(index_of_state)
                                noun_role_values.pop(index_of_state)

                            if ('purpose' in noun_roles):
                                index_of_purpose = noun_roles.index('purpose')
                                noun_roles.pop(index_of_purpose)
                                noun_role_values.pop(index_of_purpose)

                            k['noun_roles'] = noun_roles
                            k['noun_role_values'] = noun_role_values

                            available_rels = []
                            contact_rel = k.get("contacting_relationship_semantic_role_frame", {})

                            for dict in contact_rel:
                                rel_dict = dict.get("contacting_relationship")
                                if rel_dict == None:
                                    continue
                                else:
                                    available_rels.append(rel_dict)
                                    frame_dict=dict.get("frame", {})
                                    r_roles = []
                                    r_vals = []
                                    roles = frame_dict.keys()
                                    vals = frame_dict.values()
                                    r_roles.extend(roles)
                                    r_vals.extend(vals)
                                    r_vals = [self.rel_sr_val.index(r) for r in r_vals]
                                    relation_roles.append(r_roles)
                                    relation_role_values.append(r_vals)
                            k['relationships'] =  available_rels
                            k['relation_roles'] = relation_roles
                            k['relation_role_values']  = relation_role_values
                            k['contacting_relationship'] = torch.tensor([self.contacting_relationships.index(r) for r in available_rels], dtype=torch.long)
                            person = person_bbox[j]
                            gt_annotation_frame.append(k)

                    if gt_annotation_frame != []:

                        gt_annotation_frame.append({'person_bbox': person_bbox[j]['bbox']})

                        person_roles = []
                        person_role_values = []
                        atts = person_bbox[j]['attributes']
                        person_roles.extend(atts.keys())
                        person_role_values.extend(atts.values())
                        person_role_values = [self.noun_sr_val.index(r) for r in person_role_values]
                        gt_annotation_frame.append({'person_roles': person_roles})
                        gt_annotation_frame.append({'person_role_values': person_role_values})

                        gt_annotation_video.append(gt_annotation_frame)
                    else:
                        print("Removed {} as it has one contacting relationship and it is 'other relationship'".format(j))
                        video.remove(j)
            else:
                continue

            if len(video) > 2:
                self.video_list.append(video)
                self.video_size.append(person_bbox[j]['bbox_size'])
                self.gt_annotations.append(gt_annotation_video)
            elif len(video) == 1:
                self.one_frame_video += 1
            else:
                self.non_person_video += 1

        print('x'*60)
        if filter_nonperson_box_frame:
            print('There are {} videos and {} valid frames'.format(len(self.video_list), self.valid_nums))
            print('{} videos are invalid (no person), remove them'.format(self.non_person_video))
            print('{} videos are invalid (only one frame), remove them'.format(self.one_frame_video))
            print('{} frames have no human bbox in GT, remove them!'.format(self.non_gt_human_nums))
        else:
            print('There are {} videos and {} valid frames'.format(len(self.video_list), self.valid_nums))
            print('{} frames have no human bbox in GT'.format(self.non_gt_human_nums))
            print('Removed {} of them without joint heatmaps which means FasterRCNN also cannot find the human'.format(non_heatmap_nums))
        print('x' * 60)

    def __getitem__(self, index):

        frame_names = self.video_list[index]
        processed_ims = []
        im_scales = []
        raw_images_pil = []
        raw_images_np = []

        for idx, name in enumerate(frame_names):

            ImageFile.LOAD_TRUNCATED_IMAGES = True
            img_path = os.path.join(self.frames_path, name)

            if not os.path.exists(img_path):
                img_path = img_path.replace(".png", ".jpg")

            img = Image.open(img_path)
            raw_images_pil.append(img)
            im= self.preprocess(img)

            im = np.array(im)
            im=imageio.core.util.Array(im)
            im = im[:, :, ::-1] # rgb -> bgr
            im = np.transpose(im, [1, 2, 0])
            im, im_scale = prep_im_for_blob(im, [[[0.48145466, 0.4578275, 0.40821073]]], 600, 1000) #cfg.PIXEL_MEANS, target_size,

            im_scales.append(im_scale)
            processed_ims.append(im)
            raw_images_np.append(name)

        blob = im_list_to_blob(processed_ims)
        im_info = np.array([[blob.shape[1], blob.shape[2], im_scales[0]]],dtype=np.float32)
        im_info = torch.from_numpy(im_info).repeat(blob.shape[0], 1)
        img_tensor = torch.from_numpy(blob)
        img_tensor = img_tensor.permute(0, 3, 1, 2)

        gt_boxes = torch.zeros([img_tensor.shape[0], 1, 5])
        num_boxes = torch.zeros([img_tensor.shape[0]], dtype=torch.int64)

        return img_tensor, im_info, gt_boxes, num_boxes, index, raw_images_pil, raw_images_np

    def __len__(self):
        return len(self.video_list)

def cuda_collate_fn(batch):
    """
    don't need to zip the tensor

    """
    return batch[0]

Writing dataset.py


In [29]:
import sys
# Clear cached references to your local files
if 'config' in sys.modules: del sys.modules['config']
if 'dataset' in sys.modules: del sys.modules['dataset']

In [30]:
from dataset import AG

print("dataset import successful")

dataset import successful


# feature_extraction.py

In [65]:
%%writefile feature_extraction.py
import os
import gc
import numpy as np
np.set_printoptions(precision=4)
import torch
import torch.nn.functional as F
import clip
import open_clip
from utils import get_visually_prompted_features, get_visually_prompted_features_person
from config import parse_args
args = parse_args()

class FeatureExtractor:
    def organize_input(self, im_info, gt_annotation):
        bbox_num = 0
        im_idx = []
        pair = []
        a_rel = []
        s_rel = []
        c_rel = []
        rel_sr = [] # rel sem roles
        noun_sr = [] # obj sem roles
        noun_sr_values = []
        rel_sr_values = []
        nouns=[]
        relationships = []
        person_roles = []
        person_role_values = []
        video_frames = []
        person_box = []

        for i in gt_annotation:
            bbox_num += len(i)

        FINAL_BBOXES = torch.zeros([bbox_num,5], dtype=torch.float32).to(args.device)
        FINAL_LABELS = torch.zeros([bbox_num], dtype=torch.int64).to(args.device)
        FINAL_SCORES = torch.ones([bbox_num], dtype=torch.float32).to(args.device)
        HUMAN_IDX = torch.zeros([len(gt_annotation),1], dtype=torch.int64).to(args.device)

        bbox_idx = 0
        for i, j in enumerate(gt_annotation):
            for m in j:
                if 'person_roles' in m.keys():
                    person_roles.append(m['person_roles'])
                elif 'person_role_values' in m.keys():
                    person_role_values.append(m['person_role_values'])
                elif 'person_box' in m.keys():
                    person_box.append(m['person_box'])
                    FINAL_BBOXES[bbox_idx, 1:] = torch.from_numpy(m['person_box'][0])
                    FINAL_BBOXES[bbox_idx, 0] = i
                    FINAL_LABELS[bbox_idx] = 1
                    HUMAN_IDX[i] = bbox_idx
                    bbox_idx += 1
                # FIXED HERE: Added a safe condition checking for the presence of 'bbox'
                elif 'bbox' in m.keys():
                    FINAL_BBOXES[bbox_idx, 1:] = torch.from_numpy(m['bbox'])
                    FINAL_BBOXES[bbox_idx, 0] = i
                    FINAL_LABELS[bbox_idx] = m['class'] if 'class' in m.keys() else 0
                    im_idx.append(i)
                    pair.append([int(HUMAN_IDX[i]), bbox_idx])
                    a_rel.append(m['attention_relationship'].tolist() if 'attention_relationship' in m.keys() else [])
                    s_rel.append(m['spatial_relationship'].tolist() if 'spatial_relationship' in m.keys() else [])
                    c_rel.append(m['contacting_relationship'].tolist() if 'contacting_relationship' in m.keys() else [])
                    noun_sr.append(m['noun_roles'] if 'noun_roles' in m.keys() else [])
                    noun_sr_values.append(m['noun_role_values'] if 'noun_role_values' in m.keys() else [])
                    nouns.append(m['nouns'] if 'nouns' in m.keys() else "object")
                    rel_sr.append(m['relation_roles'] if 'relation_roles' in m.keys() else [])
                    rel_sr_values.append(m['relation_role_values'] if 'relation_role_values' in m.keys() else [])
                    relationships.append(m['relationships'] if 'relationships' in m.keys() else [])
                    bbox_idx += 1
                else:
                    # Silently ignore annotations that are missing key tracking fields
                    pass

        # Trim tensors down to actual collected bounding box counts to prevent empty tailing zeros mismatch
        FINAL_BBOXES = FINAL_BBOXES[:bbox_idx]
        FINAL_LABELS = FINAL_LABELS[:bbox_idx]
        FINAL_SCORES = FINAL_SCORES[:bbox_idx]

        pair = torch.tensor(pair).to(args.device)
        im_idx = torch.tensor(im_idx, dtype=torch.float).to(args.device)
        FINAL_BBOXES[:, 1:] = FINAL_BBOXES[:, 1:] / im_info[0, 2]

        entry = {
            'boxes': FINAL_BBOXES,
            'labels': FINAL_LABELS,
            'scores': FINAL_SCORES,
            'im_idx': im_idx,
            'pair_idx': pair,
            'human_idx': HUMAN_IDX,
            'attention_gt': a_rel,
            'spatial_gt': s_rel,
            'contacting_gt': c_rel,
            'noun_sr': noun_sr,
            'noun_sr_values': noun_sr_values,
            'relation_sr':rel_sr,
            'relation_sr_values':rel_sr_values,
            'relationships' : relationships,
            'nouns' : nouns,
            'person_roles' : person_roles,
            'person_role_values' : person_role_values,
            'person_box':person_box
        }
        return entry

    def get_features(self, im_info, gt_annotation, raw_images_pil, model, preprocess):
        # print("step1")
        entry = self.organize_input(im_info, gt_annotation)
        max_dim_0 = 7
        gt_padding_value = 365
        #### Create object SR features
        noun_frame_lengths = []
        n_roles = []
        v_frame_lengths = []
        v_roles = []
        obj_sr_mask = []
        verb_sr_mask = []
        class_names = entry['nouns']
        noun_roles = entry['noun_sr']
        noun_role_values = entry['noun_sr_values']
        # print("step2")
        noun_frame_lengths.extend([len(sublist) for sublist in noun_role_values])
        n_roles.extend(frame for frame in noun_roles)
        # Object SR mask
        frame_lengths = [len(sublist) for sublist in noun_role_values]
        obj_sr_mask = [[1] * (length) + [0] * (max_dim_0 - (length)) for length in frame_lengths]
        obj_sr_mask = [torch.tensor(sublist) for sublist in obj_sr_mask]
        obj_sr_mask = torch.stack(obj_sr_mask, dim=0).cpu()
        # print("step3")
        # Visually prompted image features for object sr
        boxes=entry['boxes']
        pair_idx = entry['pair_idx']
        im_idx = entry['im_idx']
        obj_box_tensor=torch.cat((im_idx[:, None], boxes[:, 1:5][pair_idx[:, 1]]), 1)
        obj_box_tensor=torch.floor(obj_box_tensor)
        obj_box_tensor=obj_box_tensor.type(torch.int64)

        # print("step4")
        image_features = get_visually_prompted_features(raw_images_pil, obj_box_tensor, model, preprocess)

        # print("step4-1")
        with torch.no_grad():
            # features for object class names
            # print("step4-2")
            class_input = clip.tokenize(class_names).to(args.device)
            # print("step4-3")
            class_features = model.encode_text(class_input)
            # features for object roles
            obj_role_features = []
            # print("model visual:", next(model.visual.parameters()).device)
            # print("token_embedding:", model.token_embedding.weight.device)
            # print("args.device:", args.device)

            for idx, item in enumerate(noun_roles):
                # print(f"Role set {idx}: {item}")
                role_input = clip.tokenize(item).to(args.device)
                # print("tokenized:", role_input.shape, role_input.device)
                with torch.no_grad():
                    role_inputs = model.encode_text(role_input)
                # print("encoded:", role_inputs.shape, role_inputs.device)
                obj_role_features.append(role_inputs)
        # print("step5")
        obj_role_features = [F.pad(tensor, (0, 0, 0, max_dim_0 - tensor.shape[0])) for tensor in obj_role_features]
        # Object SR features
        obj_sr_features = []
        for img, cls, obj_roles in zip(image_features, class_features,  obj_role_features):
            obj_roles1 = list(torch.unbind(obj_roles, dim=0))
            obj_roles1.insert(0, img)
            obj_roles1.insert(1, cls)
            feat = torch.stack(obj_roles1)
            obj_sr_features.append(feat)
        # print("step6")
        obj_sr_features = torch.stack(obj_sr_features).cpu().to(torch.float32)
        # Object sr gt
        obj_sr_gt = entry['noun_sr_values']
        obj_sr_gt = [sublist + [gt_padding_value] * (7 - len(sublist)) if len(sublist) < 7 else sublist[:7] for sublist in obj_sr_gt]
        obj_sr_gt = [torch.tensor(sublist) for sublist in obj_sr_gt]
        obj_sr_gt = torch.stack(obj_sr_gt)

        #### Create verb predicate features
        # Visuualy prompted verb predicate features
        # print("step7")
        boxes=entry['boxes']
        pair_idx = entry['pair_idx']
        im_idx = entry['im_idx']
        union_box_tensor=torch.cat((im_idx[:, None], torch.min(boxes[:, 1:3][pair_idx[:, 0]], boxes[:, 1:3][pair_idx[:, 1]]),
                                         torch.max(boxes[:, 3:5][pair_idx[:, 0]], boxes[:, 3:5][pair_idx[:, 1]])), 1)
        union_box_tensor=torch.floor(union_box_tensor)
        union_box_tensor=union_box_tensor.type(torch.int64)
        # print("step8")
        verb_image_features = get_visually_prompted_features(raw_images_pil, union_box_tensor, model, preprocess)

        verb_img_features = torch.unbind(verb_image_features, dim=0)
        verb_cls_features = torch.unbind(class_features, dim=0)
        verb_img_features = torch.stack(verb_img_features, dim=0).cpu().to(torch.float32)
        verb_cls_features = torch.stack(verb_cls_features, dim=0).cpu().to(torch.float32)
        # Verb predicate gt
        verb_gt = entry['contacting_gt']
        verb_gt = [sublist[0] for sublist in verb_gt] # one verb per object
        verb_gt = torch.tensor(verb_gt, dtype=torch.long)
        # print("step9")
        #### Create verb predicate SR features
        r_roles = entry['relation_sr']
        rel_role_values = entry['relation_sr_values']

        v_frame_lengths.extend([len(sublist[0]) for sublist in rel_role_values]) # takes only one verb SR per obj when >1 verb sr for same p-o pair
        v_roles.extend(frame[0] for frame in r_roles)
        # verb predicate SR mask
        verb_sr_frame_lengths = [len(sublist[0]) for sublist in rel_role_values]
        verb_sr_mask = [[1] * (length) + [0] * (max_dim_0 - (length)) for length in verb_sr_frame_lengths] # +2 = img_feat, obj name
        verb_sr_mask = [torch.tensor(sublist) for sublist in verb_sr_mask]
        verb_sr_mask = torch.stack(verb_sr_mask, dim=0).cpu()
        # print("step10")
        # Create visually prompted verb predicate SR features
        boxes=entry['boxes']
        pair_idx = entry['pair_idx']
        im_idx = entry['im_idx']
        union_box_tensor=torch.cat((im_idx[:, None], torch.min(boxes[:, 1:3][pair_idx[:, 0]], boxes[:, 1:3][pair_idx[:, 1]]),
                                         torch.max(boxes[:, 3:5][pair_idx[:, 0]], boxes[:, 3:5][pair_idx[:, 1]])), 1)
        union_box_tensor=torch.floor(union_box_tensor)
        union_box_tensor=union_box_tensor.type(torch.int64)
        verb_sr_image_features = get_visually_prompted_features(raw_images_pil, union_box_tensor, model, preprocess)
        # print("step11")
        with torch.no_grad():
            # verb roles
            rel_role_features = []
            for frames in r_roles:
                role_input = clip.tokenize(frames[0]).to(args.device)
                role_inputs = model.encode_text(role_input)
                rel_role_features.append(role_inputs)
        rel_role_features = [F.pad(tensor, (0, 0, 0, max_dim_0 - tensor.shape[0])) for tensor in rel_role_features]
        verb_sr_features = []
        for img, cls, verb_roles in zip(verb_sr_image_features, class_features,  rel_role_features):
            verb_roles1 = list(torch.unbind(verb_roles, dim=0))
            verb_roles1.insert(0, img)
            verb_roles1.insert(1, cls)
            feat = torch.stack(verb_roles1)
            verb_sr_features.append(feat)
        # print("step12")
        # Verb SR features
        verb_sr_features = torch.stack(verb_sr_features).cpu().to(torch.float32)
        # Verb predicate SR gt
        verb_sr_gt = rel_role_values
        verb_sr_gt = [sublist[0] for sublist in verb_sr_gt]
        verb_sr_gt = [sublist + [gt_padding_value] * (7 - len(sublist)) if len(sublist) < 7 else sublist[:7] for sublist in verb_sr_gt]
        verb_sr_gt = [torch.tensor(sublist) for sublist in verb_sr_gt]
        verb_sr_gt = torch.stack(verb_sr_gt)
        # print("step13")
        features = {}
        features['obj_sr_features'] = obj_sr_features.to(args.device)
        features['obj_sr_mask'] = obj_sr_mask.to(args.device)
        features['obj_sr_gt'] = obj_sr_gt.to(args.device)
        features['verb_img_features'] = verb_img_features.to(args.device)
        features['verb_cls_features'] = verb_cls_features.to(args.device)
        features['verb_gt'] = verb_gt.to(args.device)
        features['verb_sr_features'] = verb_sr_features.to(args.device)
        features['verb_sr_mask'] = verb_sr_mask.to(args.device)
        features['verb_sr_gt'] = verb_sr_gt.to(args.device)
        features['noun_frame_lengths'] = noun_frame_lengths
        features['n_roles'] = n_roles
        features['v_frame_lengths'] = v_frame_lengths
        features['v_roles'] = v_roles
        # print("step14")
        return features

    def get_person_features(self, im_info, gt_annotation, raw_images_pil, model, preprocess):
        # print("step4-1-1")
        entry = self.organize_input(im_info, gt_annotation)
        # print("step4-1-2")
        person_roles = entry['person_roles']
        person_role_values = entry['person_role_values']
        person_cls = ["person" for _ in range(len(person_role_values))]

        person_frame_lengths = []
        p_roles = []
        # print("step4-1-3")
        person_frame_lengths.extend([len(sublist) for sublist in person_role_values])
        p_roles.extend(frame for frame in person_roles)
        # person sr gt
        # print("step4-1-4")
        person_sr_gt = [torch.tensor(sublist) for sublist in person_role_values]
        person_sr_gt = torch.stack(person_sr_gt)

        person_box_tensor = torch.tensor(entry['person_box'])
        person_box_tensor = person_box_tensor.squeeze()
        # print("step4-1-5")
        image_features = get_visually_prompted_features_person(raw_images_pil, person_box_tensor,model, preprocess)

        with torch.no_grad():
            class_input = clip.tokenize(person_cls).to(args.device)
            class_features = model.encode_text(class_input)
            person_role_features = []
            for item in person_roles:
                role_input = clip.tokenize(item).to(args.device)
                role_inputs = model.encode_text(role_input)
                person_role_features.append(role_inputs)
        person_sr_features = []
        for img, cls, person_roles in zip(image_features, class_features,  person_role_features):
            person_roles = list(torch.unbind(person_roles, dim=0))
            person_roles.insert(0, img)
            person_roles.insert(1, cls)
            feat = torch.stack(person_roles)
            person_sr_features.append(feat)
        person_sr_features = torch.stack(person_sr_features).to(torch.float32)
        person_features = {}
        person_features['person_sr_features'] = person_sr_features.to(args.device)
        person_features['person_sr_gt'] = person_sr_gt.to(args.device)
        person_features['person_frame_lengths'] = person_frame_lengths
        person_features['p_roles'] = p_roles

        return person_features

Overwriting feature_extraction.py


In [66]:
import sys
for mod in ['config', 'dataset', 'feature_extraction']:
    if mod in sys.modules:
        del sys.modules[mod]
print("Module cache refreshed! Ready to re-run.")

Module cache refreshed! Ready to re-run.


In [32]:
from feature_extraction import FeatureExtractor

print("feature extractor import successful")

feature extractor import successful


# model.py

In [33]:
%%writefile model.py
import torch
import torch.nn as nn
from typing import Optional
from torch import Tensor, Generator
import numpy as np

from config import parse_args

args = parse_args()

def getPositionEncoding(seq_len, d):
    n=10000
    P = torch.zeros((seq_len, d))
    for k in range(seq_len):
        for i in np.arange(int(d/2)):
            denominator = np.power(n, 2*i/d)
            P[k, 2*i] = np.sin(k/denominator)
            P[k, 2*i+1] = np.cos(k/denominator)

    return P


def drop_path(x, drop_prob: float = 0.0, training: bool = False):
    """
    Stochastic Depth per sample.
    """
    if drop_prob == 0.0 or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (
        x.ndim - 1
    )  # work with diff dim tensors, not just 2D ConvNets
    mask = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    mask.floor_()  # binarize
    output = x.div(keep_prob) * mask
    return output

def normal_(self, mean: float=0, std: float=1, *, generator: Optional[Generator]=None) -> Tensor: ...


class DropPath(nn.Module):
    """Drop paths (Stochastic Depth) per sample  (when applied in main path of residual blocks)."""

    def __init__(self, drop_prob=None):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)


class Mlp(nn.Module):
    def __init__(
        self,
        in_features,
        hidden_features=None,
        out_features=None,
        act_layer=nn.GELU,
        drop=0.0,
    ):
        super().__init__()
        self.drop_rate = drop
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        if self.drop_rate > 0.0:
            self.drop = nn.Dropout(self.drop_rate)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        if self.drop_rate > 0.0:
            x = self.drop(x)
        x = self.fc2(x)
        if self.drop_rate > 0.0:
            x = self.drop(x)
        return x


class CrossAttention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.head_dim = head_dim

        self.q = nn.Linear(dim, dim, bias=qkv_bias)
        self.kv = nn.Linear(dim, dim*2, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, y, mask=None):

        B1, N1, C = x.shape # bs, roles, dimension
        B2, N2, C = y.shape # bs, image_tokens, dimension

        q = self.q(x).reshape(
            B1, N1, 1,
            self.num_heads,
            C // self.num_heads #self.num_heads
        ).permute(2, 0, 3, 1, 4)

        kv = self.kv(y).reshape(
            B2, N2, 2,
            self.num_heads,
            C // self.num_heads
        ).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]

        # Cross-Attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B1, N1, C)

        x = self.proj(x)

        if mask is not None:
            if B1 > 1:

                x = mask*x
            else:

                x = mask.unsqueeze(1).float()*x

            return x, y, attn
        else:
            return x, y, attn


class CrossAttentionBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=None,
        mlp_ratio=4.0,
        qkv_bias=False,
        drop_rate=0.0,
        drop_path=0.2,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm,
    ):
        super().__init__()
        self.dim = dim
        self.norm1 = norm_layer(dim)
        self.norm2 = norm_layer(dim)
        self.attn = CrossAttention(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
        )

        self.drop_path = (
            DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        )

        self.norm3 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            out_features=dim,
            act_layer=act_layer,
            drop=drop_rate,
        )

    def forward(self, x, y, mask=None):

        x_block, y, attn = self.attn(
                    self.norm1(x),
                    self.norm2(y),
                    mask=mask
                )
        x_norm = self.norm3(x_block)
        x_mlp = self.mlp(x_norm)
        x = x + self.drop_path(x_mlp)
        return x, y, attn


class MultiheadAttentionLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1, batch_first=True):
        super(MultiheadAttentionLayer, self).__init__()
        self.multihead_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=batch_first)
        self.layer_norm = nn.LayerNorm(embed_dim)

    def forward(self, query, key, value, key_padding_mask=None, need_weights=True, attn_mask=None):
        attn_output, attn_weights = self.multihead_attention(query, key, value, key_padding_mask=key_padding_mask, need_weights=need_weights, attn_mask=attn_mask)
        output = self.layer_norm(attn_output)
        return output, attn_weights


class InComNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.num_ans_classes = [364,17, 125]
        num_layers = 4
        num_heads = 4

        if args.clip_model == 'ViT-B/32':
            self.proj_dim = 512
        elif args.clip_model == 'ViT_L_14_336' or args.clip_model == 'ViT_L_14_336_sft':
            self.proj_dim = 768

        # obj SR
        self.obj_sr_q_proj = nn.Linear(self.proj_dim, self.proj_dim)
        self.obj_sr_kv_proj = nn.Linear(self.proj_dim, self.proj_dim)
        self.obj_sr_output_proj = nn.Linear(self.proj_dim, self.proj_dim)
        self.obj_sr_classifier = nn.Linear(self.proj_dim, self.num_ans_classes[0])

        self.obj_sr_q = nn.Parameter(torch.zeros(1,7,self.proj_dim))
        self.obj_sr_q.data.normal_(mean=0.0, std=0.02)

        self.obj_sr_pos_emb = getPositionEncoding(7, self.proj_dim)

        obj_sr_attention_layer = MultiheadAttentionLayer(embed_dim=self.proj_dim, num_heads=num_heads, dropout=0.1, batch_first=True)
        self.obj_sr_atts = nn.ModuleList([obj_sr_attention_layer for i in range(num_layers)])

        # verb
        self.verb_q_proj = nn.Linear(self.proj_dim, self.proj_dim)
        self.verb_kv_proj = nn.Linear(self.proj_dim, self.proj_dim)
        self.verb_output_proj = nn.Linear(self.proj_dim, self.proj_dim)
        self.verb_classifier = nn.Linear(self.proj_dim, self.num_ans_classes[1])
        self.verb_cls =  nn.Parameter(torch.randn(1, 1, self.proj_dim))

        self.verb_q = nn.Parameter(torch.zeros(1,1,self.proj_dim))
        self.verb_q.data.normal_(mean=0.0, std=0.02)

        verb_attention_layer = MultiheadAttentionLayer(embed_dim=self.proj_dim, num_heads=num_heads, dropout=0.1, batch_first=True)
        self.verb_atts = nn.ModuleList([verb_attention_layer for i in range(num_layers)])


        # verb SR
        self.verb_sr_q_proj = nn.Linear(self.proj_dim, self.proj_dim)
        self.verb_sr_kv_proj = nn.Linear(self.proj_dim, self.proj_dim)
        self.verb_sr_output_proj = nn.Linear(self.proj_dim, self.proj_dim)
        self.verb_sr_classifier = nn.Linear(self.proj_dim, self.num_ans_classes[2])

        self.verb_sr_q = nn.Parameter(torch.zeros(1, 7, self.proj_dim))
        self.verb_sr_q.data.normal_(mean=0.0, std=0.02)

        self.verb_sr_pos_emb = getPositionEncoding(7, self.proj_dim)

        verb_sr_attention_layer = MultiheadAttentionLayer(embed_dim=self.proj_dim, num_heads=num_heads, dropout=0.1, batch_first=True)
        self.verb_sr_atts = nn.ModuleList([verb_sr_attention_layer for i in range(num_layers)])

    def forward(self, obj_sr_flag, verb_flag, verb_sr_flag, obj_sr_frame, obj_sr_mask, verb, verb_mask, verb_sr_frame, verb_sr_mask):

        #### Object SR
        if (obj_sr_flag and not verb_flag and not verb_sr_flag):

            obj_sr_kv = obj_sr_frame
            obj_sr_kv = self.obj_sr_kv_proj(obj_sr_kv)
            obj_sr_q = self.obj_sr_q.repeat(obj_sr_frame.size(0), 1, 1)

            self.obj_sr_pos_emb = self.obj_sr_pos_emb.to(args.device)
            obj_sr_q = obj_sr_q + self.obj_sr_pos_emb

            if obj_sr_mask is not None:
                obj_sr_mask = obj_sr_mask.unsqueeze(-1).repeat(1,1,self.proj_dim)

            for layer in self.obj_sr_atts:
                obj_sr_q, obj_sr_attn = layer(obj_sr_q, obj_sr_kv, obj_sr_kv)

            obj_sr_q = obj_sr_q*obj_sr_mask
            obj_sr_q = self.obj_sr_output_proj(obj_sr_q)
            obj_sr_logits = self.obj_sr_classifier(obj_sr_q)

            return obj_sr_logits, obj_sr_q

        #### Verb predicate
        if (not obj_sr_flag and verb_flag and not verb_sr_flag):

            verb_kv = verb
            verb_kv = self.verb_kv_proj(verb_kv)
            verb_q = self.verb_q.repeat(verb.size(0), 1, 1)

            verb_mask = None
            for layer in self.verb_atts:
                verb_q, verb_attn = layer(verb_q, verb_kv, verb_kv)

            verb_q = self.verb_output_proj(verb_q)
            verb_logits = self.verb_classifier(verb_q)

            return verb_logits, verb_q

        #### Verb predicate SR
        if (not obj_sr_flag and not verb_flag and verb_sr_flag):

            verb_sr_kv = verb_sr_frame
            verb_sr_kv = self.verb_sr_kv_proj(verb_sr_kv)
            verb_sr_q = self.verb_sr_q.repeat(verb_sr_frame.size(0), 1, 1)

            self.verb_sr_pos_emb = self.verb_sr_pos_emb.to(args.device)
            verb_sr_q = verb_sr_q + self.verb_sr_pos_emb

            if verb_sr_mask is not None:
                verb_sr_mask = verb_sr_mask.unsqueeze(-1).repeat(1,1,self.proj_dim)

            for layer in self.verb_sr_atts:
                verb_sr_q, verb_sr_attn = layer(verb_sr_q, verb_sr_kv, verb_sr_kv)

            verb_sr_q = verb_sr_q*verb_sr_mask
            verb_sr_q = self.verb_sr_output_proj(verb_sr_q)
            verb_sr_logits = self.verb_sr_classifier(verb_sr_q)

            return verb_sr_logits, verb_sr_q


class InComNetPerson(nn.Module):

    def __init__(self):
        super().__init__()

        self.num_ans_classes = [364,17, 125]
        num_layers = 4
        num_heads = 4

        if args.clip_model == 'ViT-B/32':
            self.proj_dim = 512
        elif args.clip_model == 'ViT_L_14_336' or args.clip_model == 'ViT_L_14_336_sft':
            self.proj_dim = 768

        self.pos_emb = torch.nn.Parameter(torch.randn(7, self.proj_dim)) # learned pos emb


        self.person_sr_kv_proj = nn.Linear(self.proj_dim, self.proj_dim)
        self.person_sr_output_proj = nn.Linear(self.proj_dim, self.proj_dim)
        self.person_sr_classifier = nn.Linear(self.proj_dim, self.num_ans_classes[0])

        self.person_sr_q = nn.Parameter(torch.zeros(1,5,self.proj_dim))
        self.person_sr_q.data.normal_(mean=0.0, std=0.02) # std = 1

        self.person_sr_q_pos_emb = getPositionEncoding(5, self.proj_dim)
        # self.person_sr_kv_pos_emb = getPositionEncoding(7, self.proj_dim)

        person_sr_attention_layer = MultiheadAttentionLayer(embed_dim=self.proj_dim, num_heads=num_heads, dropout=0.1, batch_first=True)
        self.person_sr_atts = nn.ModuleList([person_sr_attention_layer for i in range(num_layers)])

    def forward(self, person_sr_feat):

        person_sr_kv = self.person_sr_kv_proj(person_sr_feat)
        person_sr_q = self.person_sr_q.repeat(person_sr_feat.size(0), 1, 1)
        person_sr_q = person_sr_q + self.person_sr_q_pos_emb.to(args.device)

        for layer in self.person_sr_atts:
            person_sr_q, person_sr_attn = layer(person_sr_q, person_sr_kv, person_sr_kv)

        person_sr_q = self.person_sr_output_proj(person_sr_q)
        person_sr_logits = self.person_sr_classifier(person_sr_q)

        return person_sr_logits, person_sr_q


Writing model.py


# Verification till model

In [34]:
from model import InComNet
from model import InComNetPerson

print("model import successful")

model import successful


Verify required data files

In [35]:
import os

DATA_PATH = "/content/drive/MyDrive/project/SSG/data"

files = [
    "object_classes.txt",
    "relationship_classes.txt",
    "roles.txt",
    "noun_role_values.txt",
    "relation_role_values.txt",
    "ag_ssg_combined_person.pkl",
    "ssg_dataset.pkl"
]

for f in files:
    print(f, os.path.exists(os.path.join(DATA_PATH, f)))

object_classes.txt True
relationship_classes.txt True
roles.txt True
noun_role_values.txt True
relation_role_values.txt True
ag_ssg_combined_person.pkl True
ssg_dataset.pkl True


Create AG dataset

In [36]:
import torch
import open_clip

model_name = "ViT-L-14-336"

print("Initializing OpenCLIP framework architecture...")
# 1. Pass an empty string to pretrained to create the structure without baseline weights
model, _, preprocess = open_clip.create_model_and_transforms(
    model_name=model_name,
    pretrained=""
)

# 2. Map your fine-tuned weights directly into the GPU memory
print(f"Loading custom fine-tuned weights from checkpoint...")
checkpoint = torch.load(CLIP_CKPT, map_location="cuda:0")

if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
elif isinstance(checkpoint, dict) and "model" in checkpoint:
    state_dict = checkpoint["model"]
else:
    state_dict = checkpoint

# Clean DDP prefixes if present
if any(k.startswith('module.') for k in state_dict.keys()):
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

# 3. Load the weights, push to GPU, and set to evaluation mode
model.load_state_dict(state_dict)
model = model.to("cuda:0")
model.eval()

print("SUCCESS: Model ready and mounted on GPU for feature extraction!")

Initializing OpenCLIP framework architecture...
Loading custom fine-tuned weights from checkpoint...
SUCCESS: Model ready and mounted on GPU for feature extraction!


In [37]:
from dataset import AG

dataset = AG(
    mode="train",
    datasize="large",
    data_path="/content/drive/MyDrive/project/SSG/data/",
    frame_path="/content/drive/MyDrive/project/SSG/data/frames/",
    filter_nonperson_box_frame=True,
    filter_small_box=False,
    preprocess=preprocess
)

print(len(dataset))

-------loading annotations---------slowly-----------
Removed CGNBJ.mp4/000959.png as it has one contacting relationship and it is 'other relationship'
Removed 004QE.mp4/000273.png as it has one contacting relationship and it is 'other relationship'
Removed 004QE.mp4/000343.png as it has one contacting relationship and it is 'other relationship'
Removed BOC1T.mp4/000055.png as it has one contacting relationship and it is 'other relationship'
Removed BOC1T.mp4/000379.png as it has one contacting relationship and it is 'other relationship'
Removed BOC1T.mp4/000271.png as it has one contacting relationship and it is 'other relationship'
Removed BOC1T.mp4/000497.png as it has one contacting relationship and it is 'other relationship'
Removed 05124.mp4/000603.png as it has one contacting relationship and it is 'other relationship'
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
There are 30 videos and 632 valid frames
2 videos are invalid (no person), remove them
0 videos are in

Verify dataset[0]

In [38]:
sample = dataset[0]

print(type(sample))
print(len(sample))

<class 'tuple'>
7


In [39]:
from pathlib import Path

video_dir = Path("/content/drive/MyDrive/project/SSG/data/frames/013SD.mp4")

print(video_dir.exists())

files = sorted(list(video_dir.iterdir()))[:5]
for f in files:
    print(f.name)

True
000010.jpg
000012.jpg
000027.jpg
000035.jpg
000044.jpg


In [40]:
img_tensor, im_info, gt_boxes, num_boxes, idx, raw_images_pil, raw_images_np = sample

print("img_tensor:", img_tensor.shape)
print("im_info:", im_info.shape)
print("gt_boxes:", gt_boxes.shape)
print("num_boxes:", num_boxes.shape)
print("idx:", idx)
print("num frames:", len(raw_images_pil))

img_tensor: torch.Size([19, 3, 600, 600])
im_info: torch.Size([19, 3])
gt_boxes: torch.Size([19, 1, 5])
num_boxes: torch.Size([19])
idx: 0
num frames: 19


Load  fine-tuned checkpoint:

# Clear Module
if needed

In [46]:
import sys
for mod in ['config', 'dataset', 'feature_extraction', 'utils']:
    if mod in sys.modules: del sys.modules[mod]
print("Module cache refreshed!")

Module cache refreshed!


In [47]:
import torch
import config
import utils
from feature_extraction import FeatureExtractor

# 1. Double check device registration maps to your NVIDIA L4
args = config.parse_args()
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
args.device = device

if hasattr(utils, 'args'):
    utils.args.device = device

# 2. Instantiate your extractor
extractor = FeatureExtractor()
print("Running feature extraction pass...")

# 3. Call get_features using the 'model' variable initialized in cell [24]
features = extractor.get_features(
    im_info=sample[1],
    gt_annotation=dataset.gt_annotations[sample[4]],
    raw_images_pil=sample[5],  # Your list of 19 PIL images from sample
    model=model,               # Your successfully loaded GPU checkpoint model
    preprocess=preprocess      # Your active validation transform
)

print("\nFeatures extracted successfully!")
print(f"Output shape/type: {type(features)}")

Running feature extraction pass...
step1
step2
step3
step4
step4-1
step4-2
step4-3
model visual: cuda:0
token_embedding: cuda:0
args.device: cuda:0
Role set 0: ['material', 'source', 'goal', 'affordance', 'location']
tokenized: torch.Size([5, 77]) cuda:0
encoded: torch.Size([5, 768]) cuda:0
Role set 1: ['material', 'source', 'goal', 'affordance', 'location']
tokenized: torch.Size([5, 77]) cuda:0
encoded: torch.Size([5, 768]) cuda:0
Role set 2: ['material', 'source', 'goal', 'affordance', 'location']
tokenized: torch.Size([5, 77]) cuda:0
encoded: torch.Size([5, 768]) cuda:0
Role set 3: ['material', 'source', 'goal', 'affordance', 'location']
tokenized: torch.Size([5, 77]) cuda:0
encoded: torch.Size([5, 768]) cuda:0
Role set 4: ['material', 'source', 'goal', 'affordance', 'location']
tokenized: torch.Size([5, 77]) cuda:0
encoded: torch.Size([5, 768]) cuda:0
Role set 5: ['material', 'source', 'goal', 'affordance', 'location']
tokenized: torch.Size([5, 77]) cuda:0
encoded: torch.Size([5, 7

Sanity Check

In [48]:
for k, v in features.items():
    if torch.is_tensor(v):
        print(k, v.shape, v.device)
    else:
        print(k, type(v), len(v))

obj_sr_features torch.Size([21, 9, 768]) cuda:0
obj_sr_mask torch.Size([21, 7]) cuda:0
obj_sr_gt torch.Size([21, 7]) cuda:0
verb_img_features torch.Size([21, 768]) cuda:0
verb_cls_features torch.Size([21, 768]) cuda:0
verb_gt torch.Size([21]) cuda:0
verb_sr_features torch.Size([21, 9, 768]) cuda:0
verb_sr_mask torch.Size([21, 7]) cuda:0
verb_sr_gt torch.Size([21, 7]) cuda:0
noun_frame_lengths <class 'list'> 21
n_roles <class 'list'> 21
v_frame_lengths <class 'list'> 21
v_roles <class 'list'> 21


validating the network forward pass

In [49]:
from model import InComNet

net = InComNet().cuda()
net.eval()

obj_logits, obj_q = net(
    True, False, False,
    features["obj_sr_features"],
    features["obj_sr_mask"],
    None, None,
    None, None
)

print("obj_logits:", obj_logits.shape)
print("obj_q:", obj_q.shape)

obj_logits: torch.Size([21, 7, 364])
obj_q: torch.Size([21, 7, 768])


validate the verb branch:

In [50]:
verb_feat = torch.cat(
    [
        features["verb_img_features"].unsqueeze(1),
        features["verb_cls_features"].unsqueeze(1),
        obj_q
    ],
    dim=1
)

verb_logits, verb_q = net(
    False, True, False,
    None, None,
    verb_feat, None,
    None, None
)

print("verb_logits:", verb_logits.shape)
print("verb_q:", verb_q.shape)

verb_logits: torch.Size([21, 1, 17])
verb_q: torch.Size([21, 1, 768])


validate verb SR:

In [51]:
verb_sr_feat = torch.cat(
    [
        verb_q,
        features["verb_sr_features"]
    ],
    dim=1
)

verb_sr_logits, verb_sr_q = net(
    False, False, True,
    None, None,
    None, None,
    verb_sr_feat,
    features["verb_sr_mask"]
)

print("verb_sr_logits:", verb_sr_logits.shape)
print("verb_sr_q:", verb_sr_q.shape)

verb_sr_logits: torch.Size([21, 7, 125])
verb_sr_q: torch.Size([21, 7, 768])


**Validation Result**

**Dataset**

✓ object_classes.txt

✓ relationship_classes.txt

✓ roles.txt

✓ ag_ssg_combined_person.pkl

✓ ssg_dataset.pkl

✓ frame paths fixed

✓ test dataset loads

**CLIP**

✓ ViT-L-14-336 architecture
✓ Fine-tuned checkpoint loads
✓ Image encoder works
✓ Text encoder works
✓ Visual prompting works
✓ 768-dimensional embeddings

**Feature Extraction**

✓ Object SR features

✓ Verb features

✓ Verb SR features

✓ Masks

✓ Ground-truth tensors

✓ All tensors on cuda:0

**InComNet**

✓ Object SR branch

✓ Verb branch

✓ Verb SR branch

✓ Tensor dimensions match

✓ Forward pass succeeds


**Pipeline Till Now**

Action Genome

      ↓

SSG annotations

      ↓

CLIP feature extraction

      ↓

InComNet feature construction

      ↓

InComNet forward pass

# **Training Part**

Single training

In [52]:
from model import InComNet
import torch.nn as nn

net = InComNet().cuda()

criterion = nn.CrossEntropyLoss(
    ignore_index=365,
    reduction='none'
)

optimizer = torch.optim.Adamax(
    net.parameters(),
    lr=1e-3
)

First Object SR loss:

In [53]:
obj_logits, obj_q = net(
    True, False, False,
    features["obj_sr_features"],
    features["obj_sr_mask"],
    None, None,
    None, None
)

obj_logits = obj_logits.view(-1, 364)
obj_gt = features["obj_sr_gt"].view(-1)

obj_loss = criterion(obj_logits, obj_gt)

print(obj_loss.mean())

tensor(4.3288, device='cuda:0', grad_fn=<MeanBackward0>)


In [54]:
obj_loss.mean().backward()
optimizer.step()

In [55]:
args.data_path
args.frame_path
args.clip_sft_path
args.device
args.save_path

'/content/drive/MyDrive/project/SSG/incomnet_trained_models'

In [56]:
args.data_path = "/content/drive/MyDrive/project/SSG/data/"
args.frame_path = "/content/drive/MyDrive/project/SSG/data/frames/"
args.clip_sft_path = CLIP_CKPT
args.device = "cuda:0"
args.save_path = "/content/drive/MyDrive/project/SSG/incomnet_ckpts"

In [57]:
args.datasize = "mini"
args.nepochs = 1

# **Train.py**

In [63]:
%%writefile train.py

import time
import os
import copy

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ExponentialLR
import numpy as np
np.set_printoptions(precision=3)
import wandb
import json

import clip
import open_clip

from model import InComNet
from feature_extraction import FeatureExtractor
from dataset import AG, cuda_collate_fn
from evaluation import EvaluateInComNet
from config import parse_args

import warnings
warnings.filterwarnings('ignore')

def train(args):

    torch.manual_seed(args.seed)

    if not os.path.exists(args.save_path):
        os.mkdir(args.save_path)

    # Automatically identify if we are testing a tiny set first
    is_test_run = getattr(args, 'datasize', 'large') == 'mini' or getattr(args, 'nepochs', 30) == 1
    if is_test_run:
        print("\n🚀 [FAST DEV RUN] Activated! Truncating training loops to 2 batches to test integrity.")

    # =========================================================================
    # ADDED: PIPELINE PROVENANCE LOGGING
    # =========================================================================
    print("\n" + "=" * 60)
    print("📋 ACTIVE PIPELINE RUNTIME LOGS")
    print("=" * 60)
    print(f"📊 DATASET PROFILE : {getattr(args, 'datasize', 'NOT SPECIFIED').upper()} dataset split")
    print(f"🛠️ EXTRACTION MODE : {getattr(args, 'mode', 'NOT SPECIFIED')}")

    # Track the fine-tuning checkpoint path
    sft_path = getattr(args, 'clip_sft_path', None)
    if args.clip_model == 'ViT_L_14_336_sft' and sft_path:
        print(f"💾 CLIP CHECKPOINT : {sft_path}")
    else:
        print(f"💾 CLIP CHECKPOINT : Using baseline OpenAI weights ({args.clip_model})")
    print("=" * 60 + "\n")
    # =========================================================================

    #### Initialize wandb
    if args.wandb:
        run = wandb.init(
            mode='online',
            project="InComNet",
            name="InComNet-ViT-L-14-336-sft",
            config={
                "learning_rate": args.lr,
                "epochs": args.nepochs,
            },
        )

    #### Choose CLIP model
    if args.clip_model == 'ViT_B_32':
        model, preprocess = clip.load('ViT-B/32', args.device)
    elif args.clip_model == 'ViT_L_14_336':
        model, preprocess = clip.load('ViT-L/14@336px', args.device)
    elif args.clip_model == 'ViT_L_14_336_sft':
        model_name = "ViT-L-14-336"
        model, _, preprocess = open_clip.create_model_and_transforms(model_name = model_name, pretrained = args.clip_sft_path)
    model.to(args.device)

    AG_dataset_train = AG(mode="train", datasize=args.datasize, data_path=args.data_path, frame_path=args.frame_path, filter_nonperson_box_frame=True,
                        filter_small_box=False if args.mode == 'predcls' else True, preprocess = preprocess)
    dataloader_train = torch.utils.data.DataLoader(AG_dataset_train, shuffle=False, num_workers=4,
                                                collate_fn=cuda_collate_fn, pin_memory=False)
    AG_dataset_test = AG(mode="test", datasize=args.datasize, data_path=args.data_path, frame_path=args.frame_path, filter_nonperson_box_frame=True,
                        filter_small_box=False if args.mode == 'predcls' else True, preprocess = preprocess)
    dataloader_test = torch.utils.data.DataLoader(AG_dataset_test, shuffle=False, num_workers=4,
                                                collate_fn=cuda_collate_fn, pin_memory=False)

    #### Load InComNet model
    incomnet_model = InComNet()
    incomnet_model.to(args.device)
    num_params = sum(p.numel() for p in incomnet_model.parameters())

    evaluation = EvaluateInComNet()
    feat_extractor = FeatureExtractor()

    criterion = nn.CrossEntropyLoss(ignore_index=365, reduction='none')
    optimizer = torch.optim.Adamax(incomnet_model.parameters(), lr=args.lr)
    scheduler = ExponentialLR(optimizer, gamma=0.9)

    best_val_acc = -1e8
    grand_loss = []

    for epoch in range(int(args.nepochs)):
        b = 0
        tr = []

        for data in dataloader_train:
            # Quick bypass: break training loop after 2 batches if running a mini test pass
            if is_test_run and b >= 2:
                print(f"Stopping epoch training early at batch {b} for test verification.")
                break

            objsr_loss = []
            vrb_loss = []
            vrbsr_loss = []
            start = time.time()

            incomnet_model.train()

            im_info = copy.deepcopy(data[1])
            gt_annotation = AG_dataset_train.gt_annotations[data[4]]
            raw_images_pil = copy.deepcopy(data[5])

            train_features = feat_extractor.get_features(im_info, gt_annotation, raw_images_pil, model, preprocess)
            obj_sr_features = train_features['obj_sr_features']
            obj_sr_mask = train_features['obj_sr_mask']
            obj_sr_gt = train_features['obj_sr_gt']
            verb_img_features = train_features['verb_img_features']
            verb_cls_features = train_features['verb_cls_features']
            verb_gt = train_features['verb_gt']
            verb_sr_features = train_features['verb_sr_features']
            verb_sr_mask = train_features['verb_sr_mask']
            verb_sr_gt = train_features['verb_sr_gt']
            noun_frame_lengths = train_features['noun_frame_lengths']
            n_roles = train_features['n_roles']
            v_frame_lengths = train_features['v_frame_lengths']
            v_roles = train_features['v_roles']

            loss = 0
            obj_sr_loss = 0
            verb_loss = 0
            verb_sr_loss = 0

            for i in range(10):
                if i == 0:
                    # obj SR
                    obj_sr_flag, verb_flag, verb_sr_flag = True, False, False
                    obj_sr_logits, obj_sr_q = incomnet_model(obj_sr_flag, verb_flag, verb_sr_flag, obj_sr_features, obj_sr_mask, verb=None, verb_mask=None, verb_sr_frame=None, verb_sr_mask = None)
                    obj_sr_logits = obj_sr_logits.view(-1, 364)
                    obj_sr_gt = obj_sr_gt.view(-1)
                    obj_sr_loss = criterion(obj_sr_logits, obj_sr_gt)

                elif i >= 1:
                    obj_sr_flag, verb_flag, verb_sr_flag = True, False, False
                    obj_sr2_features = torch.cat([verb_q, obj_sr_features], dim=1)
                    obj_sr_logits, obj_sr_q = incomnet_model(obj_sr_flag, verb_flag, verb_sr_flag, obj_sr2_features, obj_sr_mask, verb=None, verb_mask=None, verb_sr_frame=None, verb_sr_mask = None)
                    obj_sr_logits = obj_sr_logits.view(-1, 364)
                    obj_sr_gt = obj_sr_gt.view(-1)
                    obj_sr_loss = criterion(obj_sr_logits, obj_sr_gt)

                # verb
                obj_sr_flag, verb_flag, verb_sr_flag = False, True, False
                verb_feat = torch.cat([verb_img_features.unsqueeze(1), verb_cls_features.unsqueeze(1), obj_sr_q], dim=1)
                verb_logits, verb_q = incomnet_model(obj_sr_flag, verb_flag, verb_sr_flag, obj_sr_features, obj_sr_mask, verb_feat, verb_mask=None, verb_sr_frame=None, verb_sr_mask=None)
                verb_logits = verb_logits.squeeze()

                # Reshaping fallback if batch has a single spatial pair
                if verb_logits.dim() == 1:
                    verb_logits = verb_logits.unsqueeze(0)
                verb_loss = criterion(verb_logits, verb_gt)

                # verb SR
                obj_sr_flag, verb_flag, verb_sr_flag = False, False, True
                verb_sr_feat = torch.cat([verb_q, verb_sr_features], dim=1)
                verb_mask = None
                verb_sr_logits, verb_sr_q = incomnet_model(obj_sr_flag, verb_flag, verb_sr_flag, obj_sr_features, obj_sr_mask, verb_feat, verb_mask, verb_sr_feat, verb_sr_mask)
                verb_sr_logits = verb_sr_logits.view(-1, 125)
                verb_sr_gt = verb_sr_gt.view(-1)
                verb_sr_loss = criterion(verb_sr_logits, verb_sr_gt)

                iter_loss = torch.mean(obj_sr_loss) + torch.mean(verb_loss) + torch.mean(verb_sr_loss)
                loss += iter_loss

                objsr_loss.append(torch.mean(obj_sr_loss).item())
                vrb_loss.append(torch.mean(verb_loss).item())
                vrbsr_loss.append(torch.mean(verb_sr_loss).item())

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(incomnet_model.parameters(), max_norm=5, norm_type=2)
            optimizer.step()

            tr.append(loss.item())

            time_per_batch = (time.time() - start)
            current_lr = optimizer.param_groups[0]['lr']

            if args.wandb:
                run.log({"loss": loss, "objsr_loss": np.mean(objsr_loss), "vrb_loss": np.mean(vrb_loss), "vrbsr_loss":np.mean(vrbsr_loss)})

            log_dict = {}
            log_dict['epoch'] = epoch
            log_dict['batch'] = b
            log_dict['learning_rate'] = current_lr
            log_dict['mean_loss'] = loss
            log_dict['obj_SR_loss'] = np.mean(objsr_loss)
            log_dict['verb_loss'] = np.mean(vrb_loss)
            log_dict['verb_SR_loss'] = np.mean(vrbsr_loss)

            # Print status instantly for early validation testing
            formatted = (
                        f"epoch : {log_dict['epoch']}  batch : {log_dict['batch']}  "
                        f"learning rate : {log_dict['learning_rate']:.4f}, "
                        f"loss : {log_dict['mean_loss']:.3f}, "
                        f"obj SR loss : {log_dict['obj_SR_loss']:.3f}, "
                        f"verb loss : {log_dict['verb_loss']:.3f}, "
                        f"verb SR loss : {log_dict['verb_SR_loss']:.3f}"
                    )
            print(formatted)

            b += 1
            start = time.time()

        incomnet_model.eval()
        start = time.time()

        obj_sr_pred_list, obj_sr_gt_list, verb_pred_list, verb_gt_list, verb_sr_pred_list, verb_sr_gt_list, obj_frame_lengths, obj_roles, verb_frame_lengths, verb_roles = [], [], [], [], [], [], [], [], [], []

        test_b = 0
        for data_test in dataloader_test:
            # Quick bypass: break evaluation loop after 2 batches if testing framework pipeline
            if is_test_run and test_b >= 2:
                print(f"Stopping evaluation early at batch {test_b} for testing cycle closure.")
                break

            im_info = copy.deepcopy(data_test[1])
            gt_annotation = AG_dataset_test.gt_annotations[data_test[4]]
            raw_images_pil_test = copy.deepcopy(data_test[5])

            test_features = feat_extractor.get_features(im_info, gt_annotation, raw_images_pil_test, model, preprocess)
            obj_sr_features = test_features['obj_sr_features']
            obj_sr_mask = test_features['obj_sr_mask']
            obj_sr_gt = test_features['obj_sr_gt']
            verb_img_features = test_features['verb_img_features']
            verb_cls_features = test_features['verb_cls_features']
            verb_gt = test_features['verb_gt']
            verb_sr_features = test_features['verb_sr_features']
            verb_sr_mask = test_features['verb_sr_mask']
            verb_sr_gt = test_features['verb_sr_gt']
            noun_frame_lengths = test_features['noun_frame_lengths']
            n_roles = test_features['n_roles']
            v_frame_lengths = test_features['v_frame_lengths']
            v_roles = test_features['v_roles']

            obj_frame_lengths.extend(noun_frame_lengths)
            obj_roles.extend(n_roles)
            verb_frame_lengths.extend(v_frame_lengths)
            verb_roles.extend(v_roles)

            for i in range(args.iterations):
                if i == 0:
                    obj_sr_flag, verb_flag, verb_sr_flag = True, False, False
                    obj_sr_logits, obj_sr_q = incomnet_model(obj_sr_flag, verb_flag, verb_sr_flag, obj_sr_features, obj_sr_mask, verb=None, verb_mask=None, verb_sr_frame=None, verb_sr_mask = None)

                elif i >= 1:
                    obj_sr_flag, verb_flag, verb_sr_flag = True, False, False
                    obj_sr2_features = torch.cat([verb_q, obj_sr_features], dim=1)
                    obj_sr_logits, obj_sr_q = incomnet_model(obj_sr_flag, verb_flag, verb_sr_flag, obj_sr2_features, obj_sr_mask, verb=None, verb_mask=None, verb_sr_frame=None, verb_sr_mask = None)

                # verb
                obj_sr_flag, verb_flag, verb_sr_flag = False, True, False
                verb_feat = torch.cat([verb_img_features.unsqueeze(1), verb_cls_features.unsqueeze(1), obj_sr_q], dim=1)
                verb_logits, verb_q = incomnet_model(obj_sr_flag, verb_flag, verb_sr_flag, obj_sr_features, obj_sr_mask, verb_feat, verb_mask=None, verb_sr_frame=None, verb_sr_mask=None)

                # verb SR
                obj_sr_flag, verb_flag, verb_sr_flag = False, False, True
                verb_sr_feat = torch.cat([verb_q, verb_sr_features], dim=1)
                verb_mask = None
                verb_sr_logits, verb_sr_q = incomnet_model(obj_sr_flag, verb_flag, verb_sr_flag, obj_sr_features, obj_sr_mask, verb_feat, verb_mask, verb_sr_feat, verb_sr_mask)

            obj_sr_logits = obj_sr_logits.view(-1, 364)
            obj_sr_gt = obj_sr_gt.view(-1)
            pred_obj_sr_labels = torch.argmax(obj_sr_logits, dim=1)
            obj_sr_pred_list.extend(pred_obj_sr_labels.tolist())
            obj_sr_gt_list.extend(obj_sr_gt.tolist())

            verb_logits = verb_logits.squeeze()
            if verb_logits.dim() == 1:
                verb_logits = verb_logits.unsqueeze(0)
            pred_verb_labels = torch.argmax(verb_logits, dim=1)
            verb_pred_list.extend(pred_verb_labels.tolist())
            verb_gt_list.extend(verb_gt.tolist())

            verb_sr_logits = verb_sr_logits.view(-1, 125)
            verb_sr_gt = verb_sr_gt.view(-1)
            pred_verb_sr_labels = torch.argmax(verb_sr_logits, dim=1)
            verb_sr_pred_list.extend(pred_verb_sr_labels.tolist())
            verb_sr_gt_list.extend(verb_sr_gt.tolist())
            test_b += 1

        obj_result, verb_result, verb_sr_result = evaluation.evaluate(obj_sr_gt_list, obj_sr_pred_list, obj_frame_lengths, obj_roles, verb_gt_list, verb_pred_list, verb_sr_gt_list, verb_sr_pred_list, verb_frame_lengths, verb_roles)

        print("\nObject SR result:")
        for key, value in obj_result.items():
            print(f"{key}: {value}")

        print("\nVerb result:")
        for key, value in verb_result.items():
            print(f"{key}: {value}")

        print("\nVerb SR result:")
        for key, value in verb_sr_result.items():
            print(f"{key}: {value}")

        val_acc = obj_result['Value'] + verb_result['Accuracy'] + verb_sr_result['Value']

        print("avg epoch performance : ",  val_acc)
        print("*" * 100)
        scheduler.step()

        if val_acc > best_val_acc:
            print("val_acc is: ", epoch, val_acc)
            torch.save({"state_dict": incomnet_model.state_dict()}, os.path.join(args.save_path, f"incomnet_epoch_{epoch}.tar"))
            print("save the checkpoint after {} epochs".format(epoch))
            best_val_acc = val_acc

        with open("log_incomnet.txt", "a") as f:
            f.write(f"epoch_{epoch} Object SR: ")
            f.write(json.dumps(obj_result) + "\n")
        with open("log_incomnet.txt", "a") as f:
            f.write(f"epoch_{epoch} Verb predicate: ")
            f.write(json.dumps(verb_result) + "\n")
        with open("log_incomnet.txt", "a") as f:
            f.write(f"epoch_{epoch} Verb predicate SR: ")
            f.write(json.dumps(verb_sr_result) + "\n")


if __name__ == "__main__":
    torch.cuda.empty_cache()
    t1 = time.time()

    #### Configurations
    args = parse_args()

    print("############### Configurations ###############")
    for key, value in vars(args).items():
        print(f"{key}: {value}")
    print("##############################################")

    train(args)

Overwriting train.py


A Small fix in data set.py

In [60]:
import sys

# 1. Read your existing dataset.py file into memory
with open("/content/dataset.py", "r") as f:
    lines = f.readlines()

# 2. Modify line 130 to safely check if the key exists before indexing
updated_content = ""
for line in lines:
    if "small_object[i] = object_bbox[i]" in line:
        # Wrap the assignment in a safe 'if key in dictionary' condition
        indent = line =="" or line.split("small_object")[0]
        updated_content += f"{indent}if i in object_bbox:\n{indent}    small_object[i] = object_bbox[i]\n"
    else:
        updated_content += line

# 3. Save the corrected dataset code back to the file
with open("/content/dataset.py", "w") as f:
    f.write(updated_content)

print("SUCCESS: dataset.py successfully updated with safe key lookups!")

SUCCESS: dataset.py successfully updated with safe key lookups!


In [67]:
# 1. Clear old module versions from memory cache
import sys
for mod in ['config', 'dataset', 'feature_extraction', 'train', 'model']:
    if mod in sys.modules:
        del sys.modules[mod]

# 2. Re-run your miniature verification test pass
!python train.py \
    --data_path "/content/drive/MyDrive/project/SSG/data/" \
    --frame_path "/content/drive/MyDrive/project/SSG/data/frames/" \
    --clip_sft_path "/content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints/epoch_3.pt" \
    --save_path "/content/drive/MyDrive/project/SSG/incomnet_ckpts" \
    --device "cuda:0" \
    --datasize "mini" \
    --nepochs 1

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
############### Configurations ###############
optimizer: adamax
lr: 0.001
nepochs: 1.0
mode: predcls
iterations: 10
clip_model: ViT_L_14_336_sft
clip_sft_path: /content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints/epoch_3.pt
data_path: /content/drive/MyDrive/project/SSG/data/
frame_path: /content/drive/MyDrive/project/SSG/data/frames/
datasize: mini
save_path: /content/drive/MyDrive/project/SSG/incomnet_ckpts
model_path: /content/drive/MyDrive/project/SSG/incomnet_trained_models
ckpt: /content/drive/MyDrive/project/SSG/incomnet_trained_models/incomnet-224_epoch_0.tar
ckpt_person: /content/drive/MyDrive/project/SSG/incomnet_trained_models/incomnet-224_person_epoch_0.tar
top1: True
seed: 2222
device: cuda:

Full Training

In [69]:
# 1. Clear the module runtime cache one last time
import sys
for mod in ['config', 'dataset', 'feature_extraction', 'train', 'model']:
    if mod in sys.modules:
        del sys.modules[mod]

# 2. Kick off the full, deep training cycle
!python train.py \
    --data_path "/content/drive/MyDrive/project/SSG/data/" \
    --frame_path "/content/drive/MyDrive/project/SSG/data/frames/" \
    --clip_sft_path "/content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints/epoch_3.pt" \
    --save_path "/content/drive/MyDrive/project/SSG/incomnet_ckpts" \
    --device "cuda:0" \
    --datasize "large" \
    --nepochs 30

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
############### Configurations ###############
optimizer: adamax
lr: 0.001
nepochs: 30.0
mode: predcls
iterations: 10
clip_model: ViT_L_14_336_sft
clip_sft_path: /content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints/epoch_3.pt
data_path: /content/drive/MyDrive/project/SSG/data/
frame_path: /content/drive/MyDrive/project/SSG/data/frames/
datasize: large
save_path: /content/drive/MyDrive/project/SSG/incomnet_ckpts
model_path: /content/drive/MyDrive/project/SSG/incomnet_trained_models
ckpt: /content/drive/MyDrive/project/SSG/incomnet_trained_models/incomnet-224_epoch_0.tar
ckpt_person: /content/drive/MyDrive/project/SSG/incomnet_trained_models/incomnet-224_person_epoch_0.tar
top1: True
seed: 2222
device: cud